In [2]:
import sys
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
if sys.version_info[:2] >= (3, 14):
    raise RuntimeError(
        "Wrong kernel: Python 3.14 crashes in Jupyter. "
        "Select kernel: Python (CKD Dataset 3.12) then Restart Kernel."
    )
if ".venv312" not in sys.executable:
    print("WARNING: expected .venv312 — pick kernel Python (CKD Dataset 3.12)")


Python: 3.12.6
Executable: /usr/local/bin/python3


## 0) Setup and Imports

## SUPERVISOR RUN ORDER (use only these cells)

1. `## 0) Setup and Imports` (code cell below)
2. `## 1` to `## 10` (NHANES branch)
3. `12B` -> `12C` -> `12D` -> `12G` (temporal external) -> `12E` -> `12F` -> `15` (MIMIC optimized path)
   - Temporal external only: `12B-Resume` -> `12G`
4. `14A` -> `14B` -> `14C` -> `14D` (fusion setup + 2-branch demo)
5. `16` (WESAD wearable branch — train + export probabilities)
6. `14D-final` (3-branch fusion stub after wearable artifacts exist)
7. Skip legacy duplicate `## 12` cells below the optimized block

Run **one code cell at a time** and wait for success before the next.


In [3]:
# Placeholder removed — run the Setup cell below (## 0) Setup and Imports).

In [4]:
import os

# macOS Jupyter: limit BLAS/OpenMP threads before numpy/sklearn/torch import.
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_thread_var, "1")

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from src.data.preprocess.nhanes import merge_nhanes_tables
from src.data.labels.clinical_ckd import nhanes_clinical_ckd_label
from src.data.splits.grouped_split import grouped_split_with_class_coverage, assert_no_group_leakage
from src.eval.metrics import binary_metrics_at_threshold, safe_auroc, safe_auprc
from src.eval.calibration import expected_calibration_error, brier_score_binary
from src.models.ckd_deep_models import train_tabular_mlp, train_tabular_resmlp, TabularMLP, TabularResMLP, softmax_rows
from src.models.ckd_tree_baselines import fit_rf_xgb_tabular
import src.data.loaders.ckd_loaders as _ckd_loaders

try:
    import torch
except Exception as e:
    raise RuntimeError("PyTorch is required. Install from requirements.txt") from e


def find_best_threshold(y_true, p_pred, thr_grid):
    best_thr, best_j = 0.5, -1.0
    for t in thr_grid:
        m = binary_metrics_at_threshold(y_true, p_pred, threshold=float(t))
        j = m["sensitivity_recall"] + m["specificity"] - 1.0
        if j > best_j:
            best_thr, best_j = float(t), float(j)
    return best_thr, best_j


def enrich_binary_metrics(y_true, p_pred, threshold):
    pack = binary_metrics_at_threshold(y_true, p_pred, threshold=float(threshold))
    pack["auroc"] = float(safe_auroc(y_true, p_pred))
    pack["auprc"] = float(safe_auprc(y_true, p_pred))
    pack["ece"] = float(expected_calibration_error(y_true, p_pred))
    pack["brier"] = float(brier_score_binary(y_true, p_pred))
    return pack




import pickle


def _study_dataset_root(root: Path) -> Path:
    """Shared datasets: Title Defence/Dataset, STUDY/Dataset, or data/datasets symlink."""
    title_defence = root.parent if root.name == "CKD Dataset" else root
    candidates = [
        title_defence / "Dataset",
        title_defence.parent / "Dataset",
        root / "data" / "datasets",
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    return (title_defence / "Dataset").resolve()


def resolve_mimic_hosp(root: Path) -> Path:
    study_ds = _study_dataset_root(root)
    candidates = [
        study_ds / "mimic-iv-3.1" / "hosp",
        root / "mimic-iv-3.1" / "hosp",
        root.parent / "mimic-iv-3.1" / "hosp",
        root / "CKD Dataset" / "mimic-iv-3.1" / "hosp",
    ]
    for p in candidates:
        if (p / "admissions.csv.gz").exists():
            return p
    return candidates[0]


def resolve_nhanes_csv_root(root: Path) -> Path:
    study_ds = _study_dataset_root(root)
    candidates = [
        study_ds / "nhanes_ckd" / "csv",
        root / "nhanes_ckd" / "csv",
        root / "CKD Dataset" / "nhanes_ckd" / "csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


def resolve_wesad_root(root: Path) -> Path:
    study_ds = _study_dataset_root(root)
    candidates = [
        study_ds / "WESAD",
        root / "WESAD",
        root / "CKD Dataset" / "WESAD",
    ]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


def require_prereqs(step: str, *names: str, hint: str = "") -> None:
    """Fail fast with a clear message when a step cell runs before Setup."""
    missing = [name for name in names if name not in globals()]
    if missing:
        msg = f"{step} missing prerequisites: {missing}."
        msg += f" {hint}" if hint else " Run: kernel check -> Setup and Imports first."
        raise RuntimeError(msg)


import sys

def resolve_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "src").is_dir() and (cwd / "outputs").is_dir():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "outputs").is_dir():
        return cwd.parent
    candidate = cwd / "CKD Dataset"
    if (candidate / "outputs").is_dir():
        return candidate
    return cwd


ROOT = resolve_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUT = ROOT / "outputs" / "supervisor_runs"
OUT.mkdir(parents=True, exist_ok=True)

_ckd_loaders.NHANES_CSV = resolve_nhanes_csv_root(ROOT)
_ckd_loaders.WESAD_ROOT = resolve_wesad_root(ROOT)
_mimic_hosp_resolved = resolve_mimic_hosp(ROOT)
_ckd_loaders.MIMIC_HOSP = _mimic_hosp_resolved
_ckd_loaders.MIMIC_ROOT = _mimic_hosp_resolved.parent
CHECKPOINT_PATH = OUT / "step2_mimic_checkpoint.pkl"
print("Root:", ROOT)
print("Output:", OUT)
print("MIMIC_HOSP (resolved):", resolve_mimic_hosp(ROOT))
print("NHANES csv root:", resolve_nhanes_csv_root(ROOT))
print("WESAD root:", resolve_wesad_root(ROOT))
print("Checkpoint path:", CHECKPOINT_PATH)


Root: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset
Output: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs
MIMIC_HOSP (resolved): /Users/md.shadmantahsin/Desktop/STUDY/Dataset/mimic-iv-3.1/hosp
NHANES csv root: /Users/md.shadmantahsin/Desktop/STUDY/Dataset/nhanes_ckd/csv
WESAD root: /Users/md.shadmantahsin/Desktop/STUDY/Dataset/WESAD
Checkpoint path: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_checkpoint.pkl


In [5]:
# DATA PREFLIGHT (run after setup)
mimic_ok = (resolve_mimic_hosp(ROOT) / "admissions.csv.gz").exists()
nhanes_ok = resolve_nhanes_csv_root(ROOT).exists()
wesad_ok = resolve_wesad_root(ROOT).exists()
print("MIMIC data available:", mimic_ok, "| path:", resolve_mimic_hosp(ROOT))
print("NHANES csv available:", nhanes_ok, "| path:", resolve_nhanes_csv_root(ROOT))
print("WESAD available:", wesad_ok, "| path:", resolve_wesad_root(ROOT))
if not mimic_ok:
    print("Action: place/unzip MIMIC-IV under CKD Dataset/mimic-iv-3.1/ before running 12B.")
if not nhanes_ok:
    print("Action: place NHANES csv under CKD Dataset/nhanes_ckd/csv/ before running sections 1-10.")

MIMIC data available: True | path: /Users/md.shadmantahsin/Desktop/STUDY/Dataset/mimic-iv-3.1/hosp
NHANES csv available: True | path: /Users/md.shadmantahsin/Desktop/STUDY/Dataset/nhanes_ckd/csv
WESAD available: True | path: /Users/md.shadmantahsin/Desktop/STUDY/Dataset/WESAD


## 1) Load Data From Zero

We merge NHANES demographics + biochemistry by `SEQN`.

In [6]:
CYCLE = "2013-2014"
MAX_ROWS = 5000  # set None for full cycle

df = merge_nhanes_tables(CYCLE, max_rows=MAX_ROWS)
print("Merged shape:", df.shape)
df.head(3)

Merged shape: (5000, 84)


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,LBXSPH,LBDSPHSI,LBXSTB,LBDSTBSI,LBXSTP,LBDSTPSI,LBXSTR,LBDSTRSI,LBXSUA,LBDSUASI
4119,79606.0,8.0,2.0,1.0,44.0,NaN,1.0,1.0,1.0,NaN,...,3.5,1.130,0.5,8.55,7.4,74.0,583.0,6.582,6.7,398.5
3477,78688.0,8.0,2.0,1.0,39.0,NaN,1.0,1.0,1.0,NaN,...,3.8,1.227,0.4,6.84,7.6,76.0,79.0,0.892,6.3,374.7
4322,79910.0,8.0,2.0,1.0,14.0,NaN,3.0,3.0,2.0,169.0,...,6.5,2.099,0.1,1.71,6.6,66.0,111.0,1.253,5.1,303.3


## 2) Define Clinical Label (Not Demo Label)

**Why**: clinical rule is more defensible than a median-based placeholder.

In [7]:
y_series, label_meta = nhanes_clinical_ckd_label(
    df,
    creat_col="LBXSCR",
    age_col="RIDAGEYR",
    sex_col="RIAGENDR",
    acr_col="URDACT",            # if present in your merged table
    require_both_markers=False,   # OR rule
)

df_model = df.loc[y_series.index].copy()
y = y_series.values.astype(int)

print("Rows after label requirements:", len(df_model))
print("Positive rate:", y.mean().round(4))
print("Label metadata:")
print(label_meta)

Rows after label requirements: 4702
Positive rate: 0.0655
Label metadata:
{'rows_used': 4702.0, 'egfr_lt_60_rate': 0.0655040408336878, 'used_acr': 0.0, 'acr_non_missing_rate': 0.0, 'positive_rate': 0.0655040408336878, 'rule_mode': 0.0}


## 3) Feature Engineering (Explainable and Leakage-Safe)

### Rules used
- Keep numeric columns only
- Drop identifier and obvious non-feature columns
- Drop direct label-construction columns to avoid target leakage (`LBXSCR`, `URDACT`, `RIDAGEYR`, `RIAGENDR`)
- Remove ultra-missing columns (>60% missing)
- Keep moderate dimensionality for tabular DL stability

### Feature Engineering Starts Here (NHANES)

This next code cell performs feature engineering for the NHANES branch:
- numeric feature selection,
- leakage-sensitive column removal,
- missingness-based filtering,
- and feature capping for stable modeling.

In [8]:
id_cols = ["SEQN"]
label_defining_cols = ["LBXSCR", "URDACT", "RIDAGEYR", "RIAGENDR"]

X_num = df_model.select_dtypes(include=[np.number]).copy()
drop_cols = [c for c in (id_cols + label_defining_cols) if c in X_num.columns]
X_num = X_num.drop(columns=drop_cols, errors="ignore")

missing_rate = X_num.isna().mean()
keep_cols = missing_rate[missing_rate <= 0.60].index.tolist()
X_num = X_num[keep_cols]

# optional cap to avoid very high-dimensional sparse setup in first pass
MAX_FEATS = 120
if X_num.shape[1] > MAX_FEATS:
    # choose columns with lowest missingness first
    keep_ranked = missing_rate.loc[keep_cols].sort_values().index.tolist()[:MAX_FEATS]
    X_num = X_num[keep_ranked]

print("Engineered feature matrix shape:", X_num.shape)
print("Dropped leakage-sensitive columns:", drop_cols)
X_num.head(2)

Engineered feature matrix shape: (4702, 74)
Dropped leakage-sensitive columns: ['SEQN', 'LBXSCR', 'RIDAGEYR', 'RIAGENDR']


,SDDSRVYR,RIDSTATR,RIDRETH1,RIDRETH3,RIDEXMON,DMQMILIZ,DMDBORN4,DMDCITZN,DMDEDUC2,DMDMARTL,...,LBXSPH,LBDSPHSI,LBXSTB,LBDSTBSI,LBXSTP,LBDSTPSI,LBXSTR,LBDSTRSI,LBXSUA,LBDSUASI
4119,8.0,2.0,1.0,1.0,1.0,2.0,2.0,2.0,1.0,5.0,...,3.5,1.130,0.5,8.55,7.4,74.0,583.0,6.582,6.7,398.5
3477,8.0,2.0,1.0,1.0,1.0,2.0,1.0,1.0,3.0,5.0,...,3.8,1.227,0.4,6.84,7.6,76.0,79.0,0.892,6.3,374.7


## 4) Grouped Split (Train/Val/Test)

**Why grouped**: prevents the same subject id (`SEQN`) from appearing across splits.

In [9]:
groups = df_model["SEQN"].to_numpy()
m_train, m_val, m_test = grouped_split_with_class_coverage(groups, y, test_size=0.20, val_size=0.20, random_state=42)
assert_no_group_leakage(groups, m_train, m_val, m_test)

X_train_df, X_val_df, X_test_df = X_num.loc[m_train], X_num.loc[m_val], X_num.loc[m_test]
y_train, y_val, y_test = y[m_train], y[m_val], y[m_test]

print("Train/Val/Test sizes:", X_train_df.shape, X_val_df.shape, X_test_df.shape)
print("Class balance (train/val/test):", y_train.mean().round(3), y_val.mean().round(3), y_test.mean().round(3))

Train/Val/Test sizes: (2822, 74) (940, 74) (940, 74)
Class balance (train/val/test): 0.061 0.07 0.073


## 5) Fit Preprocessing Only on Train

We fit imputer/scaler on train only, then transform val/test.

In [10]:
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_imp = imputer.fit_transform(X_train_df)
X_val_imp = imputer.transform(X_val_df)
X_test_imp = imputer.transform(X_test_df)

X_train = scaler.fit_transform(X_train_imp)
X_val = scaler.transform(X_val_imp)
X_test = scaler.transform(X_test_imp)

print("Processed arrays:", X_train.shape, X_val.shape, X_test.shape)

Processed arrays: (2822, 74) (940, 74) (940, 74)


## 6) Baseline Model: Logistic Regression

**Why baseline?** It is stable, interpretable, and sets a strong reference.

In [11]:
baseline = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
baseline.fit(X_train, y_train)

p_val_base = baseline.predict_proba(X_val)[:, 1]
p_test_base = baseline.predict_proba(X_test)[:, 1]

thr_grid = np.linspace(0.1, 0.9, 161)
best_thr, best_j = 0.5, -1.0
for t in thr_grid:
    m = binary_metrics_at_threshold(y_val, p_val_base, threshold=float(t))
    j = m["sensitivity_recall"] + m["specificity"] - 1.0
    if j > best_j:
        best_thr, best_j = float(t), float(j)

val_base = binary_metrics_at_threshold(y_val, p_val_base, threshold=best_thr)
test_base = binary_metrics_at_threshold(y_test, p_test_base, threshold=best_thr)

for pack, yy, pp in [(val_base, y_val, p_val_base), (test_base, y_test, p_test_base)]:
    pack["auroc"] = float(safe_auroc(yy, pp))
    pack["auprc"] = float(safe_auprc(yy, pp))
    pack["ece"] = float(expected_calibration_error(yy, pp))
    pack["brier"] = float(brier_score_binary(yy, pp))

print("Baseline threshold:", best_thr)
print("Baseline val accuracy:", round(val_base["accuracy"], 4))
print("Baseline test accuracy:", round(test_base["accuracy"], 4))
print("Baseline test AUROC:", round(test_base["auroc"], 4))

Baseline threshold: 0.41000000000000003
Baseline val accuracy: 0.9372
Baseline test accuracy: 0.9394
Baseline test AUROC: 0.9822


## 7) Deep Model A: Tabular MLP

**Why MLP?** Strong first deep baseline for tabular numeric data.

In [12]:
mlp_result = train_tabular_mlp(
    X_train, y_train,
    X_val, y_val,
    hidden=(128, 64),
    dropout=0.20,
    epochs=80,
    batch_size=64,
    lr=1e-3,
    weight_decay=1e-4,
    seed=42,
)

mlp = TabularMLP(n_features=X_train.shape[1], hidden=(128, 64), dropout=0.20, n_classes=2)
mlp.load_state_dict(mlp_result.model_state)
mlp.eval()

with torch.no_grad():
    p_val_mlp = softmax_rows(mlp(torch.from_numpy(X_val.astype(np.float32))).numpy())[:, 1]
    p_test_mlp = softmax_rows(mlp(torch.from_numpy(X_test.astype(np.float32))).numpy())[:, 1]

best_thr_mlp, best_j_mlp = 0.5, -1.0
for t in thr_grid:
    m = binary_metrics_at_threshold(y_val, p_val_mlp, threshold=float(t))
    j = m["sensitivity_recall"] + m["specificity"] - 1.0
    if j > best_j_mlp:
        best_thr_mlp, best_j_mlp = float(t), float(j)

val_mlp = binary_metrics_at_threshold(y_val, p_val_mlp, threshold=best_thr_mlp)
test_mlp = binary_metrics_at_threshold(y_test, p_test_mlp, threshold=best_thr_mlp)

for pack, yy, pp in [(val_mlp, y_val, p_val_mlp), (test_mlp, y_test, p_test_mlp)]:
    pack["auroc"] = float(safe_auroc(yy, pp))
    pack["auprc"] = float(safe_auprc(yy, pp))
    pack["ece"] = float(expected_calibration_error(yy, pp))
    pack["brier"] = float(brier_score_binary(yy, pp))

print("MLP best val AUC (training):", round(mlp_result.best_val_auc, 4))
print("MLP test accuracy:", round(test_mlp["accuracy"], 4))
print("MLP test AUROC:", round(test_mlp["auroc"], 4))

MLP best val AUC (training): 0.9773
MLP test accuracy: 0.9457
MLP test AUROC: 0.9784


## 8) Deep Model B: Residual MLP

**Why ResMLP?** Residual blocks often improve optimization and stability for deeper tabular nets.

In [13]:
res_result = train_tabular_resmlp(
    X_train, y_train,
    X_val, y_val,
    d_model=128,
    n_blocks=3,
    dropout=0.10,
    epochs=100,
    batch_size=64,
    lr=8e-4,
    weight_decay=1e-4,
    seed=42,
)

resmlp = TabularResMLP(n_features=X_train.shape[1], d_model=128, n_blocks=3, dropout=0.10, n_classes=2)
resmlp.load_state_dict(res_result.model_state)
resmlp.eval()

with torch.no_grad():
    p_val_res = softmax_rows(resmlp(torch.from_numpy(X_val.astype(np.float32))).numpy())[:, 1]
    p_test_res = softmax_rows(resmlp(torch.from_numpy(X_test.astype(np.float32))).numpy())[:, 1]

best_thr_res, best_j_res = 0.5, -1.0
for t in thr_grid:
    m = binary_metrics_at_threshold(y_val, p_val_res, threshold=float(t))
    j = m["sensitivity_recall"] + m["specificity"] - 1.0
    if j > best_j_res:
        best_thr_res, best_j_res = float(t), float(j)

val_res = binary_metrics_at_threshold(y_val, p_val_res, threshold=best_thr_res)
test_res = binary_metrics_at_threshold(y_test, p_test_res, threshold=best_thr_res)

for pack, yy, pp in [(val_res, y_val, p_val_res), (test_res, y_test, p_test_res)]:
    pack["auroc"] = float(safe_auroc(yy, pp))
    pack["auprc"] = float(safe_auprc(yy, pp))
    pack["ece"] = float(expected_calibration_error(yy, pp))
    pack["brier"] = float(brier_score_binary(yy, pp))

print("ResMLP best val AUC (training):", round(res_result.best_val_auc, 4))
print("ResMLP test accuracy:", round(test_res["accuracy"], 4))
print("ResMLP test AUROC:", round(test_res["auroc"], 4))

ResMLP best val AUC (training): 0.9862
ResMLP test accuracy: 0.9617
ResMLP test AUROC: 0.9709


## 8B) Tree baselines (RF + XGBoost)

Same `X_train` / `X_val` / `X_test` as above. Validation **Youden** threshold, then test metrics. Install **XGBoost** with `pip install -r requirements.txt` (if missing, RF still runs).

In [14]:
tree_out = fit_rf_xgb_tabular(
    X_train, y_train, X_val, y_val, X_test, y_test,
    thr_grid=thr_grid,
    random_state=42,
    include_xgboost=True,
)

_trf = tree_out["random_forest"]
val_rf, test_rf = _trf["val"], _trf["test"]
best_thr_rf = float(_trf["threshold"])
print("RandomForest — val AUROC:", round(val_rf["auroc"], 4), "test AUROC:", round(test_rf["auroc"], 4), "thr:", best_thr_rf)

if tree_out["xgboost"] is not None:
    _tx = tree_out["xgboost"]
    val_xgb, test_xgb = _tx["val"], _tx["test"]
    best_thr_xgb = float(_tx["threshold"])
    print("XGBoost — val AUROC:", round(val_xgb["auroc"], 4), "test AUROC:", round(test_xgb["auroc"], 4), "thr:", best_thr_xgb)
else:
    val_xgb = test_xgb = None
    best_thr_xgb = None
    print("XGBoost:", tree_out.get("xgboost_note"))


RandomForest — val AUROC: 0.9862 test AUROC: 0.9785 thr: 0.195
XGBoost — val AUROC: 0.9925 test AUROC: 0.9922 thr: 0.10500000000000001


## 9) Natural Improvement Check (No Unrealistic Jump)

Includes **logistic regression**, **Random Forest / XGBoost** (§8B), and **deep tabular** models.

We compare baseline vs deep under the **same split** and **same feature-engineering protocol**.

In [15]:
def compact_row(name, m):
    return {
        "model": name,
        "accuracy": round(float(m["accuracy"]), 4),
        "f1": round(float(m["f1"]), 4),
        "sensitivity": round(float(m["sensitivity_recall"]), 4),
        "specificity": round(float(m["specificity"]), 4),
        "auroc": round(float(m["auroc"]), 4),
        "auprc": round(float(m["auprc"]), 4),
        "ece": round(float(m.get("ece", np.nan)), 4),
    }

_rows = [
    compact_row("baseline_logreg", test_base),
    compact_row("tabular_mlp", test_mlp),
    compact_row("tabular_resmlp", test_res),
    compact_row("random_forest", tree_out["random_forest"]["test"]),
]
if tree_out["xgboost"] is not None:
    _rows.append(compact_row("xgboost", tree_out["xgboost"]["test"]))

summary = pd.DataFrame(_rows).sort_values("auroc", ascending=False)

summary


,model,accuracy,f1,sensitivity,specificity,auroc,auprc,ece
4,xgboost,0.9660,0.7975,0.9130,0.9701,0.9922,0.9317,0.0137
0,baseline_logreg,0.9394,0.6919,0.9275,0.9403,0.9822,0.8519,0.0527
3,random_forest,0.9479,0.7135,0.8841,0.9529,0.9785,0.8315,0.0268
1,tabular_mlp,0.9457,0.6871,0.8116,0.9564,0.9784,0.8262,0.0221
2,tabular_resmlp,0.9617,0.7231,0.6812,0.9839,0.9709,0.8157,0.0315


## 10) Save Reproducible Artifacts

In [16]:
artifact = {
    "cycle": CYCLE,
    "rows_input": int(df.shape[0]),
    "rows_model": int(df_model.shape[0]),
    "label_meta": {k: float(v) for k, v in label_meta.items()},
    "n_features": int(X_num.shape[1]),
    "dropped_leakage_cols": drop_cols,
    "baseline": {"val": val_base, "test": test_base, "threshold": best_thr},
    "tabular_mlp": {"val": val_mlp, "test": test_mlp, "threshold": best_thr_mlp, "best_val_auc_training": float(mlp_result.best_val_auc)},
    "tabular_resmlp": {"val": val_res, "test": test_res, "threshold": best_thr_res, "best_val_auc_training": float(res_result.best_val_auc)},
    "random_forest": {"val": val_rf, "test": test_rf, "threshold": best_thr_rf},
    "xgboost": ({"val": val_xgb, "test": test_xgb, "threshold": best_thr_xgb} if tree_out["xgboost"] is not None else None),
    "xgboost_note": tree_out.get("xgboost_note"),
}

run_path = OUT / "from_scratch_clinical_nhanes.json"
with open(run_path, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

summary_path = OUT / "from_scratch_clinical_nhanes_summary.csv"
summary.to_csv(summary_path, index=False)

summary_ext_path = OUT / "from_scratch_clinical_nhanes_summary_extended.csv"
summary.to_csv(summary_ext_path, index=False)

print("Saved:")
print("-", run_path)
print("-", summary_path)
print("-", summary_ext_path)


Saved:
- /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/from_scratch_clinical_nhanes.json
- /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/from_scratch_clinical_nhanes_summary.csv
- /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/from_scratch_clinical_nhanes_summary_extended.csv


## 11) What to Tell Supervisor

- We started from merged NHANES tables with transparent preprocessing.
- We used a clinical CKD-risk proxy label from eGFR/ACR logic.
- We prevented leakage using grouped splits and train-only preprocessing fit.
- We compared interpretable baseline vs two deep models under identical conditions.
- We saved full artifacts for reproducibility and auditability.

## 12) Step 2: MIMIC EHR Branch (Admission-level)

We now build a **within-MIMIC** cohort (no cross-dataset patient merge) and predict CKD-risk proxy from admission-level structured features.

**Design choices:**
- Link only by valid MIMIC keys (`subject_id`, `hadm_id`)
- CKD proxy label from diagnosis codes (`ICD-9: 585*`, `ICD-10: N18*`)
- Add first-pass lab statistics from `labevents` for selected kidney-relevant labs
- Keep grouped split by `subject_id` to avoid leakage across admissions

## Step 2 Run Modes (Dev vs Supervisor)

- **Development run (fast iteration)**
  - Keep `FAST_DEV = True`
  - Uses fewer admissions and fewer scanned lab rows for faster feedback

- **Supervisor/full run (final report artifacts)**
  - Set `FAST_DEV = False`
  - Re-run Step 2 cells from config through save in order

- **Suggested full-run checklist**
  - Confirm `FAST_DEV` prints as `False`
  - Verify class balance in train/val/test output
  - Verify Step2 timing logs and that run completes without chunk truncation concerns
  - Save artifacts and confirm both JSON/CSV paths are printed

## 12B) Step 2 Optimized (New Cells)

These are newly added cells so the optimized workflow is clearly visible.

- Run this block directly for Step 2 (instead of the older Step 2 cells).
- It keeps the same output artifact paths.
- Use `FAST_DEV=True` for iteration and `FAST_DEV=False` for final supervisor runs.

### Feature Engineering Starts Here (MIMIC Step 2 Optimized)

Within the next optimized Step 2 code cell, the feature engineering section is the block labeled:
`# -------------------- assemble + split + preprocess --------------------`

That block handles:
- lab merge,
- categorical one-hot encoding,
- valid-column filtering,
- train-only imputation and scaling.

In [17]:
# Step 2 optimized end-to-end block (new cell)
import time

_REQUIRED_12B = (
    "ROOT", "OUT", "CHECKPOINT_PATH", "resolve_mimic_hosp",
    "pd", "np", "json", "pickle", "torch",
    "SimpleImputer", "StandardScaler", "LogisticRegression",
    "grouped_split_with_class_coverage", "assert_no_group_leakage",
    "find_best_threshold", "enrich_binary_metrics",
    "train_tabular_mlp", "TabularMLP", "softmax_rows",
)
_missing_12b = [name for name in _REQUIRED_12B if name not in globals()]
if _missing_12b:
    raise RuntimeError(
        f"12B missing prerequisites: {_missing_12b}. "
        "Run: kernel check -> Setup and Imports -> 12B."
    )

_t_step2_all = time.perf_counter()

MIMIC_HOSP = resolve_mimic_hosp(ROOT)
assert MIMIC_HOSP.exists(), f"MIMIC hosp folder not found: {MIMIC_HOSP}"

FAST_DEV = False
N_ADMISSIONS = 15000 if FAST_DEV else 30000
LAB_CHUNKSIZE = 2_000_000
MAX_LAB_ROWS = 4_000_000 if FAST_DEV else 12_000_000

# -------------------- config + core loads --------------------
_t_cfg = time.perf_counter()
adm = pd.read_csv(
    MIMIC_HOSP / "admissions.csv.gz",
    usecols=["subject_id", "hadm_id", "admittime", "dischtime", "admission_type", "insurance", "marital_status", "race"],
    low_memory=False,
)
pat = pd.read_csv(
    MIMIC_HOSP / "patients.csv.gz",
    usecols=["subject_id", "gender", "anchor_age"],
    low_memory=False,
)
diag = pd.read_csv(
    MIMIC_HOSP / "diagnoses_icd.csv.gz",
    usecols=["hadm_id", "icd_code", "icd_version"],
    low_memory=False,
)

adm = adm.dropna(subset=["subject_id", "hadm_id"]).drop_duplicates(subset=["hadm_id"])
if len(adm) > N_ADMISSIONS:
    adm = adm.sample(N_ADMISSIONS, random_state=42)

cohort = adm.merge(pat, on="subject_id", how="left")
cohort["admittime"] = pd.to_datetime(cohort["admittime"], errors="coerce")
cohort["dischtime"] = pd.to_datetime(cohort["dischtime"], errors="coerce")
cohort["los_hours"] = (cohort["dischtime"] - cohort["admittime"]).dt.total_seconds() / 3600.0

print("Step2 cohort admissions:", cohort.shape)
print("FAST_DEV:", FAST_DEV, "| N_ADMISSIONS:", N_ADMISSIONS, "| MAX_LAB_ROWS:", MAX_LAB_ROWS)
print("Step2 config/load time (sec):", round(time.perf_counter() - _t_cfg, 2))

# -------------------- label --------------------
_t_label = time.perf_counter()
diag = diag.dropna(subset=["hadm_id", "icd_code", "icd_version"]).copy()
diag["icd_code"] = diag["icd_code"].astype(str).str.upper().str.strip()
diag["icd_version"] = pd.to_numeric(diag["icd_version"], errors="coerce")

is_ckd_icd9 = (diag["icd_version"] == 9) & diag["icd_code"].str.startswith("585")
is_ckd_icd10 = (diag["icd_version"] == 10) & diag["icd_code"].str.startswith("N18")
ckd_hadm = set(diag.loc[is_ckd_icd9 | is_ckd_icd10, "hadm_id"].astype("int64").tolist())

cohort["hadm_id"] = cohort["hadm_id"].astype("int64")
cohort["subject_id"] = cohort["subject_id"].astype("int64")
cohort["ckd_label"] = cohort["hadm_id"].isin(ckd_hadm).astype(int)

print("CKD positive rate (Step2):", round(float(cohort["ckd_label"].mean()), 4))
print("Step2 label time (sec):", round(time.perf_counter() - _t_label, 2))

# -------------------- optimized labs --------------------
_t_labs = time.perf_counter()

# Admission-relative window for lab aggregation.
# Use None to disable window filtering.
LAB_TIME_WINDOW_HOURS = 24 if FAST_DEV else 48

labitems = pd.read_csv(
    MIMIC_HOSP / "d_labitems.csv.gz",
    usecols=["itemid", "label"],
    low_memory=False,
)
labitems["label_l"] = labitems["label"].astype(str).str.lower().str.strip()

# Tighter mapping: include + exclude patterns to reduce noisy matches.
lab_rules = {
    "lab_creatinine": {
        "include": ["creatinine"],
        "exclude": ["urine", "clearance"],
    },
    "lab_urea_nitrogen": {
        "include": ["urea nitrogen", "bun"],
        "exclude": ["urine"],
    },
    "lab_potassium": {
        "include": ["potassium"],
        "exclude": ["urine", "whole blood"],
    },
    "lab_sodium": {
        "include": ["sodium"],
        "exclude": ["urine", "whole blood"],
    },
    "lab_chloride": {
        "include": ["chloride"],
        "exclude": ["urine", "whole blood"],
    },
    "lab_bicarbonate": {
        "include": ["bicarbonate", "hco3"],
        "exclude": ["urine"],
    },
    "lab_hemoglobin": {
        "include": ["hemoglobin"],
        "exclude": ["urine"],
    },
    "lab_platelet": {
        "include": ["platelet"],
        "exclude": ["urine"],
    },
}

itemid_to_feat = {}
for feat, rule in lab_rules.items():
    include_mask = pd.Series(False, index=labitems.index)
    for pat in rule["include"]:
        include_mask = include_mask | labitems["label_l"].str.contains(pat, regex=False, na=False)

    exclude_mask = pd.Series(False, index=labitems.index)
    for pat in rule["exclude"]:
        exclude_mask = exclude_mask | labitems["label_l"].str.contains(pat, regex=False, na=False)

    hit = include_mask & (~exclude_mask)
    if hit.any():
        for iid in labitems.loc[hit, "itemid"].astype("int64").tolist():
            if iid not in itemid_to_feat:
                itemid_to_feat[iid] = feat

selected_itemids = np.array(sorted(itemid_to_feat.keys()), dtype=np.int64)
cohort_hadm = np.array(sorted(cohort["hadm_id"].astype("int64").unique()), dtype=np.int64)
hadm_to_admit = cohort.set_index("hadm_id")["admittime"]

acc = []
rows_seen = 0
chunks_seen = 0
for chunk in pd.read_csv(
    MIMIC_HOSP / "labevents.csv.gz",
    usecols=["hadm_id", "itemid", "valuenum", "charttime"],
    chunksize=LAB_CHUNKSIZE,
    low_memory=False,
):
    # Enforce hard cap before processing chunk body.
    if MAX_LAB_ROWS is not None and rows_seen >= MAX_LAB_ROWS:
        break

    chunk_len = len(chunk)
    remaining = None if MAX_LAB_ROWS is None else (MAX_LAB_ROWS - rows_seen)
    if remaining is not None and remaining < chunk_len:
        chunk = chunk.iloc[:remaining].copy()
        chunk_len = len(chunk)

    chunks_seen += 1
    rows_seen += chunk_len

    x = chunk.dropna(subset=["hadm_id", "itemid", "valuenum"]).copy()
    if x.empty:
        continue

    x["itemid"] = x["itemid"].astype("int64")
    x = x[x["itemid"].isin(selected_itemids)]
    if x.empty:
        continue

    x["hadm_id"] = x["hadm_id"].astype("int64")
    x = x[x["hadm_id"].isin(cohort_hadm)]
    if x.empty:
        continue

    if LAB_TIME_WINDOW_HOURS is not None:
        x["charttime"] = pd.to_datetime(x["charttime"], errors="coerce")
        x["admittime"] = x["hadm_id"].map(hadm_to_admit)
        x = x.dropna(subset=["charttime", "admittime"])
        if x.empty:
            continue
        dt_hours = (x["charttime"] - x["admittime"]).dt.total_seconds() / 3600.0
        x = x[(dt_hours >= 0.0) & (dt_hours <= LAB_TIME_WINDOW_HOURS)]
        if x.empty:
            continue

    x["feature"] = x["itemid"].map(itemid_to_feat)
    g = x.groupby(["hadm_id", "feature"], as_index=False)["valuenum"].median()
    acc.append(g)

    if len(acc) >= 20:
        acc = [pd.concat(acc, ignore_index=True).groupby(["hadm_id", "feature"], as_index=False)["valuenum"].median()]

if acc:
    labs_long = pd.concat(acc, ignore_index=True)
    labs_long = labs_long.groupby(["hadm_id", "feature"], as_index=False)["valuenum"].median()
    labs = labs_long.pivot_table(index="hadm_id", columns="feature", values="valuenum", aggfunc="median").reset_index()
else:
    labs = pd.DataFrame({"hadm_id": cohort["hadm_id"].astype("int64")})

print("Mapped lab itemids:", len(selected_itemids))
print("Lab time window hours:", LAB_TIME_WINDOW_HOURS)
print("Chunks scanned from labevents:", chunks_seen)
print("Rows scanned from labevents:", rows_seen)
print("Lab feature frame:", labs.shape)
print("Step2 labs time (sec):", round(time.perf_counter() - _t_labs, 2))

# -------------------- assemble + split + preprocess --------------------
_t_prep = time.perf_counter()
m2 = cohort.merge(labs, on="hadm_id", how="left")
cat_cols = ["gender", "admission_type", "insurance", "marital_status", "race"]
for c in cat_cols:
    if c not in m2.columns:
        m2[c] = "UNKNOWN"

base_num_cols = ["anchor_age", "los_hours"]
for c in base_num_cols:
    if c not in m2.columns:
        m2[c] = np.nan

lab_cols = [c for c in m2.columns if str(c).startswith("lab_")]
feature_df = m2[base_num_cols + cat_cols + lab_cols].copy()
feature_df = pd.get_dummies(feature_df, columns=cat_cols, dummy_na=True)

valid_cols = [c for c in feature_df.columns if feature_df[c].notna().sum() > 50]
X2_df = feature_df[valid_cols].replace([np.inf, -np.inf], np.nan)
y2 = m2["ckd_label"].astype(int).to_numpy()
groups2 = m2["subject_id"].astype("int64").to_numpy()

m2_tr, m2_va, m2_te = grouped_split_with_class_coverage(groups2, y2, test_size=0.20, val_size=0.20, random_state=42, max_tries=100)
assert_no_group_leakage(groups2, m2_tr, m2_va, m2_te)

X2_tr_df, X2_va_df, X2_te_df = X2_df.loc[m2_tr], X2_df.loc[m2_va], X2_df.loc[m2_te]
y2_tr, y2_va, y2_te = y2[m2_tr], y2[m2_va], y2[m2_te]

imp2 = SimpleImputer(strategy="median")
sc2 = StandardScaler()
X2_tr = sc2.fit_transform(imp2.fit_transform(X2_tr_df))
X2_va = sc2.transform(imp2.transform(X2_va_df))
X2_te = sc2.transform(imp2.transform(X2_te_df))

print("Step2 X shape:", X2_df.shape)
print("Step2 split sizes:", X2_tr.shape, X2_va.shape, X2_te.shape)
print("Step2 class balance train/val/test:", round(y2_tr.mean(), 3), round(y2_va.mean(), 3), round(y2_te.mean(), 3))
print("Step2 assemble/preprocess time (sec):", round(time.perf_counter() - _t_prep, 2))

# -------------------- baseline + mlp --------------------
_t_models = time.perf_counter()
base2 = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
base2.fit(X2_tr, y2_tr)

p2_va_base = base2.predict_proba(X2_va)[:, 1]
p2_te_base = base2.predict_proba(X2_te)[:, 1]
thr_grid2 = np.linspace(0.1, 0.9, 161)

best_thr2, _ = find_best_threshold(y2_va, p2_va_base, thr_grid2)
val2_base = enrich_binary_metrics(y2_va, p2_va_base, best_thr2)
test2_base = enrich_binary_metrics(y2_te, p2_te_base, best_thr2)

mlp2_epochs = 20 if FAST_DEV else 35
mlp2_result = train_tabular_mlp(
    X2_tr, y2_tr, X2_va, y2_va,
    hidden=(128, 64), dropout=0.20,
    epochs=mlp2_epochs, batch_size=64,
    lr=1e-3, weight_decay=1e-4, seed=42,
)

mlp2 = TabularMLP(n_features=X2_tr.shape[1], hidden=(128, 64), dropout=0.20, n_classes=2)
mlp2.load_state_dict(mlp2_result.model_state)
mlp2.eval()

with torch.no_grad():
    x2_va_t = torch.from_numpy(X2_va.astype(np.float32))
    x2_te_t = torch.from_numpy(X2_te.astype(np.float32))
    p2_va_mlp = softmax_rows(mlp2(x2_va_t).numpy())[:, 1]
    p2_te_mlp = softmax_rows(mlp2(x2_te_t).numpy())[:, 1]

best_thr2_mlp, _ = find_best_threshold(y2_va, p2_va_mlp, thr_grid2)
val2_mlp = enrich_binary_metrics(y2_va, p2_va_mlp, best_thr2_mlp)
test2_mlp = enrich_binary_metrics(y2_te, p2_te_mlp, best_thr2_mlp)

step2_summary = pd.DataFrame([
    {"model": "mimic_logreg", "accuracy": test2_base["accuracy"], "f1": test2_base["f1"], "auroc": test2_base["auroc"], "auprc": test2_base["auprc"], "ece": test2_base["ece"]},
    {"model": "mimic_tabular_mlp", "accuracy": test2_mlp["accuracy"], "f1": test2_mlp["f1"], "auroc": test2_mlp["auroc"], "auprc": test2_mlp["auprc"], "ece": test2_mlp["ece"]},
]).sort_values("auroc", ascending=False)

print("Step2 MLP epochs:", mlp2_epochs)
print("Step2 baseline AUROC:", round(test2_base["auroc"], 4))
print("Step2 MLP AUROC:", round(test2_mlp["auroc"], 4))
print("Step2 model/eval time (sec):", round(time.perf_counter() - _t_models, 2))

# -------------------- save artifacts --------------------
_t_save = time.perf_counter()
step2_artifact = {
    "cohort": "mimic_hosp_admission_level",
    "n_admissions_requested": int(N_ADMISSIONS),
    "rows_model": int(len(m2)),
    "positive_rate": float(m2["ckd_label"].mean()),
    "n_features": int(X2_df.shape[1]),
    "lab_feature_cols": lab_cols,
    "baseline": {"val": val2_base, "test": test2_base, "threshold": float(best_thr2)},
    "tabular_mlp": {
        "val": val2_mlp,
        "test": test2_mlp,
        "threshold": float(best_thr2_mlp),
        "best_val_auc_training": float(mlp2_result.best_val_auc),
    },
}

step2_json = OUT / "step2_mimic_admission_branch.json"
step2_csv = OUT / "step2_mimic_admission_summary.csv"
with open(step2_json, "w", encoding="utf-8") as f:
    json.dump(step2_artifact, f, indent=2)
step2_summary.to_csv(step2_csv, index=False)

print("Saved Step2 artifacts:")
print("-", step2_json)
print("-", step2_csv)
print("Step2 save time (sec):", round(time.perf_counter() - _t_save, 2))
print("Step2 total time (sec):", round(time.perf_counter() - _t_step2_all, 2))

_step2_ckpt = {"FAST_DEV": FAST_DEV, "X2_tr": X2_tr, "X2_va": X2_va, "X2_te": X2_te, "y2_tr": y2_tr, "y2_va": y2_va, "y2_te": y2_te, "X2_df": X2_df, "m2": m2, "lab_cols": lab_cols, "thr_grid2": thr_grid2, "p2_va_base": p2_va_base, "p2_te_base": p2_te_base, "p2_va_mlp": p2_va_mlp, "p2_te_mlp": p2_te_mlp, "base2": base2, "step2_summary": step2_summary}
with open(CHECKPOINT_PATH, "wb") as _f:
    pickle.dump(_step2_ckpt, _f)
print("Saved Step2 checkpoint:", CHECKPOINT_PATH)
step2_summary


Step2 cohort admissions: (30000, 11)
FAST_DEV: False | N_ADMISSIONS: 30000 | MAX_LAB_ROWS: 12000000
Step2 config/load time (sec): 2.93
CKD positive rate (Step2): 0.1523
Step2 label time (sec): 3.28
Mapped lab itemids: 90
Lab time window hours: 48
Chunks scanned from labevents: 6
Rows scanned from labevents: 12000000
Lab feature frame: (1758, 9)
Step2 labs time (sec): 9.07
Step2 X shape: (30000, 68)
Step2 split sizes: (17994, 68) (6009, 68) (5997, 68)
Step2 class balance train/val/test: 0.151 0.148 0.162
Step2 assemble/preprocess time (sec): 0.16
Step2 MLP epochs: 35
Step2 baseline AUROC: 0.7614
Step2 MLP AUROC: 0.7663
Step2 model/eval time (sec): 5.4
Saved Step2 artifacts:
- /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_admission_branch.json
- /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_admission_summary.csv
Step2 save time (sec): 0.0
Step2 total time (sec): 20.84
Saved Step2 c

,model,accuracy,f1,auroc,auprc,ece
1,mimic_tabular_mlp,0.645323,0.405700,0.766264,0.385742,0.028363
0,mimic_logreg,0.702184,0.416721,0.761351,0.379870,0.248566


## 12B-Resume) Load Step2 Checkpoint (if needed)

Run once before `12C` if `12B` was already executed.

In [18]:
import pickle
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "rb") as _f:
        _ckpt = pickle.load(_f)
    for _k, _v in _ckpt.items():
        globals()[_k] = _v
    print("Loaded Step2 checkpoint:", CHECKPOINT_PATH)
else:
    print("No checkpoint yet. Run 12B first:", CHECKPOINT_PATH)

Loaded Step2 checkpoint: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_checkpoint.pkl


## 12C) Step 2 Operating Point Comparison (New Cell)

This cell compares three clinically useful threshold policies on the same test set:
- Youden-J optimal threshold (current default),
- Max-F1 threshold,
- Target-recall threshold (default: recall >= 0.85).

Use this table in slides to explain trade-offs rather than relying only on accuracy.

In [19]:
# Step 2 operating-point comparison (LogReg + MLP)
# Requires variables from 12B cell: y2_va, y2_te, p2_va_base, p2_te_base, p2_va_mlp, p2_te_mlp, thr_grid2

def pick_threshold_by_policy(y_true, p_pred, thr_grid, policy="youden", target_recall=0.85):
    best_thr = 0.5
    best_score = -1e18

    for t in thr_grid:
        m = binary_metrics_at_threshold(y_true, p_pred, threshold=float(t))
        if policy == "youden":
            score = m["sensitivity_recall"] + m["specificity"] - 1.0
        elif policy == "f1":
            score = m["f1"]
        elif policy == "target_recall":
            rec = m["sensitivity_recall"]
            spec = m["specificity"]
            # Prefer thresholds meeting recall target; then maximize specificity.
            score = spec if rec >= target_recall else (-1e6 + rec)
        else:
            raise ValueError(f"Unknown policy: {policy}")

        if score > best_score:
            best_score = score
            best_thr = float(t)

    return best_thr


def eval_policy_row(model_name, policy_name, y_val, p_val, y_test, p_test, thr_grid, target_recall=0.85):
    thr = pick_threshold_by_policy(y_val, p_val, thr_grid, policy=policy_name, target_recall=target_recall)
    m = enrich_binary_metrics(y_test, p_test, thr)
    return {
        "model": model_name,
        "policy": policy_name,
        "threshold": round(float(thr), 4),
        "accuracy": round(float(m["accuracy"]), 4),
        "f1": round(float(m["f1"]), 4),
        "recall": round(float(m["sensitivity_recall"]), 4),
        "specificity": round(float(m["specificity"]), 4),
        "auroc": round(float(m["auroc"]), 4),
        "auprc": round(float(m["auprc"]), 4),
        "ece": round(float(m["ece"]), 4),
    }

TARGET_RECALL = 0.85
policies = ["youden", "f1", "target_recall"]

rows = []
for pol in policies:
    rows.append(eval_policy_row("mimic_logreg", pol, y2_va, p2_va_base, y2_te, p2_te_base, thr_grid2, target_recall=TARGET_RECALL))
for pol in policies:
    rows.append(eval_policy_row("mimic_tabular_mlp", pol, y2_va, p2_va_mlp, y2_te, p2_te_mlp, thr_grid2, target_recall=TARGET_RECALL))

op_summary = pd.DataFrame(rows).sort_values(["model", "policy"]).reset_index(drop=True)
print("Target recall policy uses recall >=", TARGET_RECALL)
op_summary
op_summary_path = OUT / "step2_mimic_operating_point_summary.csv"
op_summary.to_csv(op_summary_path, index=False)
print("Saved:", op_summary_path)


Target recall policy uses recall >= 0.85
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_operating_point_summary.csv


## 12D) Step 2 Robustness via Repeated Splits (New Cell)

This cell runs repeated grouped splits and reports mean±std metrics.

Defaults are tuned for practical runtime:
- model: Logistic Regression only,
- seeds: 3 runs,
- grouped split by `subject_id`.

If needed, you can later extend this to include MLP in repeated runs.

In [20]:
# Step 2 repeated-split robustness (LogReg)
# Requires variables from 12B cell: m2, lab_cols

RSEEDS = [42, 52, 62, 72, 82]
THR_GRID_RS = np.linspace(0.1, 0.9, 161)
BOOTSTRAP_N = 2000
BOOTSTRAP_SEED = 7

cat_cols_rs = ["gender", "admission_type", "insurance", "marital_status", "race"]
base_num_cols_rs = ["anchor_age", "los_hours"]

m2_rs = m2.copy()
for c in cat_cols_rs:
    if c not in m2_rs.columns:
        m2_rs[c] = "UNKNOWN"
for c in base_num_cols_rs:
    if c not in m2_rs.columns:
        m2_rs[c] = np.nan

feature_df_rs = m2_rs[base_num_cols_rs + cat_cols_rs + lab_cols].copy()
feature_df_rs = pd.get_dummies(feature_df_rs, columns=cat_cols_rs, dummy_na=True)
valid_cols_rs = [c for c in feature_df_rs.columns if feature_df_rs[c].notna().sum() > 50]
X_rs_df = feature_df_rs[valid_cols_rs].replace([np.inf, -np.inf], np.nan)
y_rs = m2_rs["ckd_label"].astype(int).to_numpy()
groups_rs = m2_rs["subject_id"].astype("int64").to_numpy()

rows = []
for seed in RSEEDS:
    m_tr, m_va, m_te = grouped_split_with_class_coverage(
        groups_rs, y_rs,
        test_size=0.20,
        val_size=0.20,
        random_state=int(seed),
        max_tries=100,
    )
    assert_no_group_leakage(groups_rs, m_tr, m_va, m_te)

    X_tr_df, X_va_df, X_te_df = X_rs_df.loc[m_tr], X_rs_df.loc[m_va], X_rs_df.loc[m_te]
    y_tr, y_va, y_te = y_rs[m_tr], y_rs[m_va], y_rs[m_te]

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    X_tr = sc.fit_transform(imp.fit_transform(X_tr_df))
    X_va = sc.transform(imp.transform(X_va_df))
    X_te = sc.transform(imp.transform(X_te_df))

    clf = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=int(seed))
    clf.fit(X_tr, y_tr)

    p_va = clf.predict_proba(X_va)[:, 1]
    p_te = clf.predict_proba(X_te)[:, 1]
    thr, _ = find_best_threshold(y_va, p_va, THR_GRID_RS)
    m_te_pack = enrich_binary_metrics(y_te, p_te, thr)

    rows.append({
        "seed": int(seed),
        "threshold": float(thr),
        "accuracy": float(m_te_pack["accuracy"]),
        "f1": float(m_te_pack["f1"]),
        "recall": float(m_te_pack["sensitivity_recall"]),
        "specificity": float(m_te_pack["specificity"]),
        "auroc": float(m_te_pack["auroc"]),
        "auprc": float(m_te_pack["auprc"]),
        "ece": float(m_te_pack["ece"]),
    })

rs_detail = pd.DataFrame(rows)
metric_cols = ["accuracy", "f1", "recall", "specificity", "auroc", "auprc", "ece"]
rs_mean = rs_detail[metric_cols].mean()
rs_std = rs_detail[metric_cols].std(ddof=1)

rs_summary = pd.DataFrame({
    "metric": metric_cols,
    "mean": [float(rs_mean[m]) for m in metric_cols],
    "std": [float(rs_std[m]) for m in metric_cols],
})

# Bootstrap 95% CI on AUROC and F1 across per-seed test metrics
rng_bs = np.random.default_rng(BOOTSTRAP_SEED)
ci_rows = []
for metric in ["auroc", "f1"]:
    vals = rs_detail[metric].to_numpy(dtype=float)
    boots = []
    for _ in range(BOOTSTRAP_N):
        sample = rng_bs.choice(vals, size=len(vals), replace=True)
        boots.append(float(np.mean(sample)))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    ci_rows.append({
        "metric": metric,
        "mean": float(vals.mean()),
        "std": float(vals.std(ddof=1)),
        "ci95_low": float(lo),
        "ci95_high": float(hi),
        "n_seeds": int(len(vals)),
        "bootstrap_n": int(BOOTSTRAP_N),
    })

rs_ci = pd.DataFrame(ci_rows)

print("Repeated-split seeds:", RSEEDS)
print(rs_detail.round(4).to_string(index=False))
print("\nMean ± std:")
print(rs_summary.assign(mean=lambda d: d["mean"].round(4), std=lambda d: d["std"].round(4)))
print("\nBootstrap 95% CI (AUROC, F1):")
print(rs_ci.round(4).to_string(index=False))

rs_detail.to_csv(OUT / "step2_mimic_robustness_per_seed.csv", index=False)
rs_summary.to_csv(OUT / "step2_mimic_robustness_mean_std.csv", index=False)
rs_ci.to_csv(OUT / "step2_mimic_robustness_ci95.csv", index=False)
print("Saved:", OUT / "step2_mimic_robustness_per_seed.csv")
print("Saved:", OUT / "step2_mimic_robustness_ci95.csv")


Repeated-split seeds: [42, 52, 62, 72, 82]
 seed  threshold  accuracy     f1  recall  specificity  auroc  auprc    ece
   42      0.520    0.7022 0.4167  0.6584       0.7106 0.7614 0.3799 0.2486
   52      0.495    0.6861 0.4083  0.7271       0.6789 0.7713 0.3607 0.2639
   62      0.465    0.6638 0.4013  0.7712       0.6455 0.7794 0.3642 0.2651
   72      0.400    0.5906 0.3769  0.8226       0.5495 0.7677 0.3802 0.2709
   82      0.480    0.6722 0.4033  0.7286       0.6621 0.7652 0.3696 0.2580

Mean ± std:
        metric    mean     std
0     accuracy  0.6630  0.0430
1           f1  0.4013  0.0149
2       recall  0.7416  0.0607
3  specificity  0.6493  0.0608
4        auroc  0.7690  0.0069
5        auprc  0.3709  0.0089
6          ece  0.2613  0.0085

Bootstrap 95% CI (AUROC, F1):
metric   mean    std  ci95_low  ci95_high  n_seeds  bootstrap_n
 auroc 0.7690 0.0069    0.7641     0.7742        5         2000
    f1 0.4013 0.0149    0.3885     0.4113        5         2000
Saved: /Users/md.

## 12G) Temporal External Validation (MIMIC)

**External validation (temporal)** — train on earlier subjects, test on later subjects.

Split rule (subject-level, leakage-safe):

- For each `subject_id`, take the **first** `admittime` across admissions.
- **80th percentile** of those first-admission times is the temporal cutoff.
- Subjects with first admission **before** cutoff → **train**; **on or after** cutoff → **test**.
- **All admissions** of a subject stay on the same side.

Preprocessing and model fitting use **train only**; evaluation is on the temporal test set. Compare briefly to the random grouped split test AUROC from `12B` when available.

Prerequisite: **`12B-Resume`** (or `12B`) so `m2`, `X2_df`, and `lab_cols` are in memory or in `step2_mimic_checkpoint.pkl`.


In [21]:
# 12G) Temporal external validation — MIMIC admission cohort
import json
import pickle
import time
from pathlib import Path

from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

try:
    from sklearn.frozen import FrozenEstimator
except ImportError:  # sklearn < 1.8
    FrozenEstimator = None

if "OUT" not in globals():
    OUT = Path.cwd() / "outputs" / "supervisor_runs"
if "CHECKPOINT_PATH" not in globals():
    CHECKPOINT_PATH = OUT / "step2_mimic_checkpoint.pkl"

_REQUIRED_12G = ("m2", "X2_df", "lab_cols", "enrich_binary_metrics", "find_best_threshold", "OUT")
_missing_12g = [n for n in _REQUIRED_12G if n not in globals()]
if _missing_12g:
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "rb") as _f:
            _ckpt_12g = pickle.load(_f)
        for _k, _v in _ckpt_12g.items():
            globals()[_k] = _v
        print("Loaded Step2 checkpoint for 12G:", CHECKPOINT_PATH)
        _missing_12g = [n for n in _REQUIRED_12G if n not in globals()]
    if _missing_12g:
        raise RuntimeError(
            f"12G missing prerequisites: {_missing_12g}. "
            "Run: kernel check -> Setup -> 12B-Resume (or 12B) -> 12G."
        )

_t_12g = time.perf_counter()

print("=" * 72)
print("EXTERNAL VALIDATION (temporal)")
print("=" * 72)

# Align cohort metadata with feature matrix index
meta = m2[["subject_id", "admittime", "ckd_label"]].copy()
meta["admittime"] = pd.to_datetime(meta["admittime"], errors="coerce")
if not meta.index.equals(X2_df.index):
    meta = meta.loc[X2_df.index]

meta = meta.dropna(subset=["admittime", "subject_id"]).copy()
idx_ok = meta.index
X_all = X2_df.loc[idx_ok]
y_all = meta["ckd_label"].astype(int).to_numpy()

# Subject-level temporal split: first admittime per subject
subj_first = meta.groupby("subject_id", as_index=True)["admittime"].min().sort_values()
cutoff_ts = subj_first.quantile(0.80)
cutoff_str = pd.Timestamp(cutoff_ts).isoformat()

train_subj = set(subj_first.index[subj_first < cutoff_ts])
test_subj = set(subj_first.index[subj_first >= cutoff_ts])

is_train = meta["subject_id"].isin(train_subj).to_numpy()
is_test = meta["subject_id"].isin(test_subj).to_numpy()
assert not np.any(is_train & is_test)
assert len(train_subj & test_subj) == 0

m_tr = is_train
m_te = is_test

print(f"Temporal cutoff (80th percentile of subject first admittime): {cutoff_str}")
print(f"  train admissions: {int(m_tr.sum())} | test admissions: {int(m_te.sum())}")
print(f"  train subjects: {len(train_subj)} | test subjects: {len(test_subj)}")
print(
    "  first-admit range train:",
    subj_first[subj_first.index.isin(train_subj)].min(),
    "..",
    subj_first[subj_first.index.isin(train_subj)].max(),
)
print(
    "  first-admit range test:",
    subj_first[subj_first.index.isin(test_subj)].min(),
    "..",
    subj_first[subj_first.index.isin(test_subj)].max(),
)

X_tr_df = X_all.loc[m_tr]
X_te_df = X_all.loc[m_te]
y_tr, y_te = y_all[m_tr], y_all[m_te]

# Train-only imputation + scaling (do not reuse sc2/imp2 from 12B)
imp_temp = SimpleImputer(strategy="median")
sc_temp = StandardScaler()
X_tr = sc_temp.fit_transform(imp_temp.fit_transform(X_tr_df))
X_te = sc_temp.transform(imp_temp.transform(X_te_df))

THR_GRID_TEMP = thr_grid2 if "thr_grid2" in globals() else np.linspace(0.1, 0.9, 161)

base_temp = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
base_temp.fit(X_tr, y_tr)

if FrozenEstimator is not None:
    cal_temp = CalibratedClassifierCV(FrozenEstimator(base_temp), method="sigmoid", cv=3)
else:
    cal_temp = CalibratedClassifierCV(
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
        method="sigmoid",
        cv=3,
    )
cal_temp.fit(X_tr, y_tr)

p_tr_cal = cal_temp.predict_proba(X_tr)[:, 1]
p_te_cal = cal_temp.predict_proba(X_te)[:, 1]

thr_temp, _ = find_best_threshold(y_tr, p_tr_cal, THR_GRID_TEMP)
m_temp_te = enrich_binary_metrics(y_te, p_te_cal, thr_temp)

# Compare to random grouped split test AUROC (12B)
grouped_auroc = None
grouped_source = None
if "test2_base" in globals():
    grouped_auroc = float(test2_base["auroc"])
    grouped_source = "test2_base (12B in-memory)"
else:
    _s2json = OUT / "step2_mimic_admission_branch.json"
    if _s2json.exists():
        with open(_s2json, "r", encoding="utf-8") as _f:
            _s2 = json.load(_f)
        grouped_auroc = float(_s2["baseline"]["test"]["auroc"])
        grouped_source = "step2_mimic_admission_branch.json"
    elif "p2_te_base" in globals() and "y2_te" in globals():
        grouped_auroc = float(safe_auroc(y2_te, p2_te_base))
        grouped_source = "checkpoint p2_te_base"

print("\nTemporal test metrics (sigmoid-calibrated LogReg):")
for k, label in [
    ("auroc", "auroc"),
    ("auprc", "auprc"),
    ("f1", "f1"),
    ("sensitivity_recall", "recall"),
    ("specificity", "specificity"),
    ("ece", "ece"),
    ("brier", "brier"),
]:
    print(f"  {label}: {round(float(m_temp_te[k]), 4)}")

if grouped_auroc is not None:
    delta = float(m_temp_te["auroc"]) - grouped_auroc
    print(f"\nComparison — random grouped split test AUROC ({grouped_source}): {round(grouped_auroc, 4)}")
    print(f"  temporal minus grouped: {delta:+.4f} (negative = drop on future admissions)")

summary_row = {
    "split": "temporal_external_80pct_subject_first_admit",
    "cutoff_admittime": cutoff_str,
    "n_train_admissions": int(m_tr.sum()),
    "n_test_admissions": int(m_te.sum()),
    "n_train_subjects": len(train_subj),
    "n_test_subjects": len(test_subj),
    "threshold": round(float(thr_temp), 4),
    "auroc": round(float(m_temp_te["auroc"]), 4),
    "auprc": round(float(m_temp_te["auprc"]), 4),
    "f1": round(float(m_temp_te["f1"]), 4),
    "recall": round(float(m_temp_te["sensitivity_recall"]), 4),
    "specificity": round(float(m_temp_te["specificity"]), 4),
    "ece": round(float(m_temp_te["ece"]), 4),
    "brier": round(float(m_temp_te["brier"]), 4),
}
if grouped_auroc is not None:
    summary_row["grouped_split_test_auroc"] = round(grouped_auroc, 4)
    summary_row["auroc_delta_vs_grouped"] = round(float(m_temp_te["auroc"]) - grouped_auroc, 4)

temp_ext_summary = pd.DataFrame([summary_row])
temp_ext_csv = OUT / "step2_mimic_temporal_external_summary.csv"
temp_ext_summary.to_csv(temp_ext_csv, index=False)

temp_ext_json = {
    "validation_type": "temporal_external_subject_level",
    "rule": "subjects with first admittime < 80th-percentile cutoff -> train; all admissions of subject on same side",
    "cutoff_admittime": cutoff_str,
    "cutoff_percentile": 0.80,
    "n_train_admissions": int(m_tr.sum()),
    "n_test_admissions": int(m_te.sum()),
    "n_train_subjects": len(train_subj),
    "n_test_subjects": len(test_subj),
    "threshold": float(thr_temp),
    "metrics_test": {
        "auroc": float(m_temp_te["auroc"]),
        "auprc": float(m_temp_te["auprc"]),
        "f1": float(m_temp_te["f1"]),
        "recall": float(m_temp_te["sensitivity_recall"]),
        "specificity": float(m_temp_te["specificity"]),
        "ece": float(m_temp_te["ece"]),
        "brier": float(m_temp_te["brier"]),
    },
    "grouped_split_comparison": {
        "source": grouped_source,
        "test_auroc": grouped_auroc,
        "auroc_delta_temporal_minus_grouped": (
            float(m_temp_te["auroc"]) - grouped_auroc if grouped_auroc is not None else None
        ),
    },
}

temp_ext_json_path = OUT / "step2_mimic_temporal_external.json"
with open(temp_ext_json_path, "w", encoding="utf-8") as _f:
    json.dump(temp_ext_json, _f, indent=2)

print("\nSaved:", temp_ext_csv)
print("Saved:", temp_ext_json_path)
print("12G time (sec):", round(time.perf_counter() - _t_12g, 2))
temp_ext_summary


EXTERNAL VALIDATION (temporal)
Temporal cutoff (80th percentile of subject first admittime): 2178-08-26T17:44:00
  train admissions: 23874 | test admissions: 6126
  train subjects: 20948 | test subjects: 5238
  first-admit range train: 2110-01-11 10:14:00 .. 2178-08-26 16:36:00
  first-admit range test: 2178-08-26 17:44:00 .. 2214-02-01 13:15:00

Temporal test metrics (sigmoid-calibrated LogReg):
  auroc: 0.771
  auprc: 0.4236
  f1: 0.4415
  recall: 0.715
  specificity: 0.68
  ece: 0.0222
  brier: 0.1229

Comparison — random grouped split test AUROC (test2_base (12B in-memory)): 0.7614
  temporal minus grouped: +0.0097 (negative = drop on future admissions)

Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_temporal_external_summary.csv
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_temporal_external.json
12G time (sec): 0.22


,split,cutoff_admittime,n_train_admissions,n_test_admissions,n_train_subjects,n_test_subjects,threshold,auroc,auprc,f1,recall,specificity,ece,brier,grouped_split_test_auroc,auroc_delta_vs_grouped
0,temporal_external_80pct_subject_first_admit,2178-08-26T17:44:00,23874,6126,20948,5238,0.155,0.771,0.4236,0.4415,0.715,0.68,0.0222,0.1229,0.7614,0.0097


## 12E) Probability Calibration Check (New Cell)

This cell compares:
- uncalibrated Logistic Regression,
- sigmoid-calibrated Logistic Regression,
- isotonic-calibrated Logistic Regression.

Goal: reduce ECE and improve probability trustworthiness while tracking AUROC/AUPRC/F1.

In [22]:
# Calibration comparison on Step 2 (LogReg)
# Requires variables from 12B cell: X2_tr, X2_va, X2_te, y2_tr, y2_va, y2_te, p2_va_base, p2_te_base, thr_grid2

from sklearn.calibration import CalibratedClassifierCV

# Base estimator used for calibration wrappers.
base_lr_for_cal = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)

cal_sigmoid = CalibratedClassifierCV(base_lr_for_cal, method="sigmoid", cv=3)
cal_sigmoid.fit(X2_tr, y2_tr)

base_lr_for_iso = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
cal_isotonic = CalibratedClassifierCV(base_lr_for_iso, method="isotonic", cv=3)
cal_isotonic.fit(X2_tr, y2_tr)

p2_va_sig = cal_sigmoid.predict_proba(X2_va)[:, 1]
p2_te_sig = cal_sigmoid.predict_proba(X2_te)[:, 1]

p2_va_iso = cal_isotonic.predict_proba(X2_va)[:, 1]
p2_te_iso = cal_isotonic.predict_proba(X2_te)[:, 1]

thr_base, _ = find_best_threshold(y2_va, p2_va_base, thr_grid2)
thr_sig, _ = find_best_threshold(y2_va, p2_va_sig, thr_grid2)
thr_iso, _ = find_best_threshold(y2_va, p2_va_iso, thr_grid2)

m_base = enrich_binary_metrics(y2_te, p2_te_base, thr_base)
m_sig = enrich_binary_metrics(y2_te, p2_te_sig, thr_sig)
m_iso = enrich_binary_metrics(y2_te, p2_te_iso, thr_iso)

calibration_summary = pd.DataFrame([
    {
        "model": "logreg_uncalibrated",
        "threshold": round(float(thr_base), 4),
        "accuracy": round(float(m_base["accuracy"]), 4),
        "f1": round(float(m_base["f1"]), 4),
        "recall": round(float(m_base["sensitivity_recall"]), 4),
        "specificity": round(float(m_base["specificity"]), 4),
        "auroc": round(float(m_base["auroc"]), 4),
        "auprc": round(float(m_base["auprc"]), 4),
        "ece": round(float(m_base["ece"]), 4),
        "brier": round(float(m_base["brier"]), 4),
    },
    {
        "model": "logreg_sigmoid_cal",
        "threshold": round(float(thr_sig), 4),
        "accuracy": round(float(m_sig["accuracy"]), 4),
        "f1": round(float(m_sig["f1"]), 4),
        "recall": round(float(m_sig["sensitivity_recall"]), 4),
        "specificity": round(float(m_sig["specificity"]), 4),
        "auroc": round(float(m_sig["auroc"]), 4),
        "auprc": round(float(m_sig["auprc"]), 4),
        "ece": round(float(m_sig["ece"]), 4),
        "brier": round(float(m_sig["brier"]), 4),
    },
    {
        "model": "logreg_isotonic_cal",
        "threshold": round(float(thr_iso), 4),
        "accuracy": round(float(m_iso["accuracy"]), 4),
        "f1": round(float(m_iso["f1"]), 4),
        "recall": round(float(m_iso["sensitivity_recall"]), 4),
        "specificity": round(float(m_iso["specificity"]), 4),
        "auroc": round(float(m_iso["auroc"]), 4),
        "auprc": round(float(m_iso["auprc"]), 4),
        "ece": round(float(m_iso["ece"]), 4),
        "brier": round(float(m_iso["brier"]), 4),
    },
]).sort_values("ece", ascending=True)

print("Lower ECE is better calibration.")
calibration_summary
calibration_summary_path = OUT / "step2_mimic_calibration_summary.csv"
calibration_summary.to_csv(calibration_summary_path, index=False)
print("Saved:", calibration_summary_path)


Lower ECE is better calibration.
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_calibration_summary.csv


### Defense Interpretation (Calibration)

- Calibration improved strongly after post-hoc calibration (ECE dropped from high to very low), so predicted risks are more trustworthy.
- Discrimination metrics (AUROC/AUPRC) stayed nearly unchanged, which is expected because calibration mainly reshapes probability confidence, not ranking order.
- For reporting and clinical decision support, use calibrated probabilities (isotonic/sigmoid) and explicitly state the selected operating threshold policy.

## FINAL REPORTING LOCK (agreed)

- Primary MIMIC model: logreg_sigmoid_cal
- Primary threshold policy: f1
- Fusion: protocol ready; final aligned evaluation pending wearable

In [23]:
FINAL_REPORTING = {"mimic_model": "logreg_sigmoid_cal", "threshold_policy_primary": "f1", "threshold_policy_screening": "target_recall", "fusion_status": "protocol_ready_final_eval_pending"}
with open(OUT / "final_reporting_lock.json", "w", encoding="utf-8") as f:
    json.dump(FINAL_REPORTING, f, indent=2)
print("Saved:", OUT / "final_reporting_lock.json")
FINAL_REPORTING

Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/final_reporting_lock.json


{'mimic_model': 'logreg_sigmoid_cal',
 'threshold_policy_primary': 'f1',
 'threshold_policy_screening': 'target_recall',
 'fusion_status': 'protocol_ready_final_eval_pending'}

## 12F) Step 2 Tree Baselines (RF + XGBoost) — New Cell

Run **after `12B`** (needs `X2_tr`, `X2_va`, `X2_te`, `y2_*`, `thr_grid2`).

Same split and preprocessing as LogReg/MLP. Updates `step2_summary` and saved Step 2 artifacts.

In [24]:
# Step 2F: Random Forest + XGBoost on MIMIC branch (same matrices as 12B)
import time

_REQUIRED_12F = (
    "X2_tr", "X2_va", "X2_te", "y2_tr", "y2_va", "y2_te",
    "thr_grid2", "step2_summary", "OUT", "fit_rf_xgb_tabular", "pd",
)
_missing_12f = [name for name in _REQUIRED_12F if name not in globals()]
if _missing_12f:
    _hint = "Run: kernel check -> Setup -> 12B-Resume (or 12B) -> 12F."
    if "CHECKPOINT_PATH" in globals() and CHECKPOINT_PATH.exists():
        _hint = "Run: kernel check -> Setup -> 12B-Resume -> 12F."
    raise RuntimeError(f"12F missing prerequisites: {_missing_12f}. {_hint}")

_t_tree2 = time.perf_counter()

tree2_out = fit_rf_xgb_tabular(
    X2_tr, y2_tr, X2_va, y2_va, X2_te, y2_te,
    thr_grid=thr_grid2,
    random_state=42,
    include_xgboost=True,
    n_jobs=1,
)

_trf2 = tree2_out["random_forest"]
test2_rf = _trf2["test"]
print(
    "MIMIC RandomForest — test AUROC:", round(test2_rf["auroc"], 4),
    "| AUPRC:", round(test2_rf["auprc"], 4),
    "| F1:", round(test2_rf["f1"], 4),
    "| thr:", round(float(_trf2["threshold"]), 4),
)

rows_tree2 = [
    {
        "model": "mimic_random_forest",
        "accuracy": test2_rf["accuracy"],
        "f1": test2_rf["f1"],
        "auroc": test2_rf["auroc"],
        "auprc": test2_rf["auprc"],
        "ece": test2_rf["ece"],
    }
]

if tree2_out["xgboost"] is not None:
    _tx2 = tree2_out["xgboost"]
    test2_xgb = _tx2["test"]
    print(
        "MIMIC XGBoost — test AUROC:", round(test2_xgb["auroc"], 4),
        "| AUPRC:", round(test2_xgb["auprc"], 4),
        "| F1:", round(test2_xgb["f1"], 4),
        "| thr:", round(float(_tx2["threshold"]), 4),
    )
    rows_tree2.append({
        "model": "mimic_xgboost",
        "accuracy": test2_xgb["accuracy"],
        "f1": test2_xgb["f1"],
        "auroc": test2_xgb["auroc"],
        "auprc": test2_xgb["auprc"],
        "ece": test2_xgb["ece"],
    })
else:
    print("XGBoost skipped:", tree2_out.get("xgboost_note"))

step2_summary_ext = pd.concat([
    step2_summary,
    pd.DataFrame(rows_tree2),
], ignore_index=True).sort_values("auroc", ascending=False)

print("Step2 tree-baseline time (sec):", round(time.perf_counter() - _t_tree2, 2))
step2_summary_ext
step2_ext_csv = OUT / "step2_mimic_admission_summary_extended.csv"
step2_summary_ext.to_csv(step2_ext_csv, index=False)
print("Saved extended summary:", step2_ext_csv)


MIMIC RandomForest — test AUROC: 0.7676 | AUPRC: 0.3668 | F1: 0.4227 | thr: 0.44
MIMIC XGBoost — test AUROC: 0.7683 | AUPRC: 0.3818 | F1: 0.4085 | thr: 0.43
Step2 tree-baseline time (sec): 4.69
Saved extended summary: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_admission_summary_extended.csv


## 12H) Rebuild Step2 Report Tables from Saved JSON (fallback)

Use this if raw MIMIC files are unavailable but `step2_mimic_admission_branch.json` already exists from a prior successful `12B` run.


In [25]:
# Rebuild supervisor tables from saved Step2 JSON (no retrain)
step2_json = OUT / "step2_mimic_admission_branch.json"
assert step2_json.exists(), f"Missing {step2_json}. Run 12B first on a machine with MIMIC data."

with open(step2_json, "r", encoding="utf-8") as f:
    s2 = json.load(f)

report_rows = []
for model_key, label in [("baseline", "mimic_logreg"), ("tabular_mlp", "mimic_tabular_mlp")]:
    t = s2[model_key]["test"]
    report_rows.append({
        "model": label,
        "accuracy": t["accuracy"],
        "f1": t["f1"],
        "auroc": t["auroc"],
        "auprc": t["auprc"],
        "ece": t["ece"],
        "threshold": s2[model_key]["threshold"],
    })

step2_report_from_json = pd.DataFrame(report_rows).sort_values("auroc", ascending=False)
print("Rebuilt Step2 test summary from JSON:")
step2_report_from_json

step2_report_csv = OUT / "step2_mimic_report_from_json.csv"
step2_report_from_json.to_csv(step2_report_csv, index=False)
print("Saved:", step2_report_csv)

Rebuilt Step2 test summary from JSON:
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_report_from_json.csv


## 15) Step 2 Explainability (Feature Importance) — New Cell

Run **after `12B`** and preferably after **`12E`** (uses calibrated LogReg if available).

Permutation importance on a held-out sample (fast, no extra SHAP install). Top features support XAI narrative for supervisor.

In [26]:
# Step 2 explainability: permutation importance + case-level examples
from sklearn.inspection import permutation_importance

_t_xai = time.perf_counter()

if "cal_sigmoid" in globals():
    xai_model = cal_sigmoid
    xai_model_name = "logreg_sigmoid_cal"
else:
    xai_model = base2
    xai_model_name = "mimic_logreg_uncalibrated"

n_xai = 800 if FAST_DEV else 2000
rng = np.random.default_rng(42)
idx_xai = rng.choice(len(X2_te), size=min(n_xai, len(X2_te)), replace=False)
X_xai = X2_te[idx_xai]
y_xai = y2_te[idx_xai]

perm = permutation_importance(
    xai_model,
    X_xai,
    y_xai,
    n_repeats=5,
    random_state=42,
    scoring="roc_auc",
    n_jobs=-1,
)

feat_names = list(X2_df.columns)
imp_df = pd.DataFrame({
    "feature": feat_names,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

top_n = 15
xai_top = imp_df.head(top_n).reset_index(drop=True)
print("Explainability model:", xai_model_name)
print("Sample size:", len(X_xai))
xai_top

xai_csv = OUT / "step2_mimic_permutation_importance_top15.csv"
xai_top.to_csv(xai_csv, index=False)
print("Saved:", xai_csv)

# Case-level examples (1 high-risk TP, 1 low-risk TN) on calibrated LogReg
if "cal_sigmoid" in globals():
    p_te_xai = cal_sigmoid.predict_proba(X2_te)[:, 1]
else:
    p_te_xai = base2.predict_proba(X2_te)[:, 1]

thr_xai, _ = find_best_threshold(y2_va, cal_sigmoid.predict_proba(X2_va)[:, 1] if "cal_sigmoid" in globals() else p2_va_base, thr_grid2)
pred_te = (p_te_xai >= thr_xai).astype(int)

case_rows = []
for label_name, mask_fn in [
    ("high_risk_true_positive", lambda y, p, pred: (y == 1) & (pred == 1)),
    ("low_risk_true_negative", lambda y, p, pred: (y == 0) & (pred == 0)),
]:
    idx_all = np.arange(len(y2_te))
    m = mask_fn(y2_te, p_te_xai, pred_te)
    if not np.any(m):
        continue
    cand = idx_all[m]
    pick = int(cand[np.argmax(p_te_xai[cand])]) if label_name.startswith("high") else int(cand[np.argmin(p_te_xai[cand])])
    row = {
        "case_id": label_name,
        "model": xai_model_name,
        "hadm_index": int(pick),
        "y_true": int(y2_te[pick]),
        "p_ckd": float(p_te_xai[pick]),
        "threshold": float(thr_xai),
        "pred": int(pred_te[pick]),
        "top_feature_1": str(imp_df.iloc[0]["feature"]),
    }
    top3 = imp_df.head(3)["feature"].tolist()
    for j, f in enumerate(top3, start=1):
        row[f"global_top_feature_{j}"] = f
    case_rows.append(row)

case_df = pd.DataFrame(case_rows)
case_csv = OUT / "step2_mimic_case_examples.csv"
case_df.to_csv(case_csv, index=False)
print("Saved:", case_csv)
case_df

# Optional SHAP (skipped if import/install too heavy for this run)
shap_status = {"model": xai_model_name, "status": "skipped", "reason": None}
try:
    import shap  # noqa: F401
    shap_status["reason"] = "shap available but skipped to keep notebook runtime light; use permutation + case table for Freeze #4"
except Exception as exc:
    shap_status["reason"] = f"shap not used: {exc}"
with open(OUT / "step2_mimic_shap_status.json", "w", encoding="utf-8") as f:
    json.dump(shap_status, f, indent=2)
print("SHAP status:", shap_status)
print("XAI block time (sec):", round(time.perf_counter() - _t_xai, 2))


Explainability model: logreg_sigmoid_cal
Sample size: 2000
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_permutation_importance_top15.csv
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_case_examples.csv


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP status: {'model': 'logreg_sigmoid_cal', 'status': 'skipped', 'reason': 'shap available but skipped to keep notebook runtime light; use permutation + case table for Freeze #4'}
XAI block time (sec): 10.7


## 15B) SHAP for MIMIC LogReg (one cell)

Run **after `12B-Resume`** (or after `12B` + `12E`). Lightweight Linear SHAP on the reporting model.

In [27]:
# 15B) SHAP — logreg_sigmoid_cal (reporting model)
import pickle
import shap

_t_shap = time.perf_counter()

# Load Step2 matrices if kernel was restarted
if "X2_te" not in globals() and CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "rb") as _f:
        _ck = pickle.load(_f)
    for _k, _v in _ck.items():
        if _k != "step2_summary":
            globals()[_k] = _v
    print("Loaded checkpoint for SHAP:", CHECKPOINT_PATH)

assert "X2_te" in globals(), "Run 12B or 12B-Resume first."

# Reporting model: calibrated sigmoid logreg (refit if 12E not in memory)
if "cal_sigmoid" not in globals():
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.frozen import FrozenEstimator
    cal_sigmoid = CalibratedClassifierCV(FrozenEstimator(base2), method="sigmoid")
    cal_sigmoid.fit(X2_va, y2_va)
    print("Fitted cal_sigmoid on validation for SHAP.")

feat_names = list(X2_df.columns)
bg_n, ex_n = 200, 400
rng = np.random.default_rng(42)
bg_idx = rng.choice(len(X2_tr), size=min(bg_n, len(X2_tr)), replace=False)
ex_idx = rng.choice(len(X2_te), size=min(ex_n, len(X2_te)), replace=False)

# Explain underlying linear model (same features as calibrated logreg)
explainer = shap.LinearExplainer(base2, X2_tr[bg_idx], feature_perturbation="interventional")
shap_vals = explainer.shap_values(X2_te[ex_idx])
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]  # positive class

mean_abs = np.abs(shap_vals).mean(axis=0)
shap_df = pd.DataFrame({
    "feature": feat_names,
    "mean_abs_shap": mean_abs,
}).sort_values("mean_abs_shap", ascending=False)

shap_top = shap_df.head(15).reset_index(drop=True)
print("SHAP model: logreg_sigmoid_cal (linear explainer on base2)")
print("Background / explained:", len(bg_idx), "/", len(ex_idx))
shap_top

shap_csv = OUT / "step2_mimic_shap_top15.csv"
shap_top.to_csv(shap_csv, index=False)

shap_status = {
    "model": "logreg_sigmoid_cal",
    "status": "completed",
    "explainer": "LinearExplainer",
    "background_n": int(len(bg_idx)),
    "explained_n": int(len(ex_idx)),
    "output_csv": str(shap_csv.name),
}
with open(OUT / "step2_mimic_shap_status.json", "w", encoding="utf-8") as f:
    json.dump(shap_status, f, indent=2)

print("Saved:", shap_csv)
print("Saved:", OUT / "step2_mimic_shap_status.json")
print("SHAP time (sec):", round(time.perf_counter() - _t_shap, 2))


SHAP model: logreg_sigmoid_cal (linear explainer on base2)
Background / explained: 200 / 400
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_shap_top15.csv
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_shap_status.json
SHAP time (sec): 0.01


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/shap/explainers/_linear.py:99: FutureWarning: The feature_perturbation option is now deprecated in favor of using the appropriate masker (maskers.Independent, maskers.Partition or maskers.Impute).
  warnings.warn(wmsg, FutureWarning)


In [28]:
# Step 2 config + core table loads
_t_step2_cfg = time.perf_counter()

MIMIC_HOSP = ROOT / "mimic-iv-3.1" / "hosp"
assert MIMIC_HOSP.exists(), f"MIMIC hosp folder not found: {MIMIC_HOSP}"

# Fast defaults for notebook iteration; flip to False for fuller runs.
FAST_DEV = True
N_ADMISSIONS = 15000 if FAST_DEV else 30000
LAB_CHUNKSIZE = 2_000_000
MAX_LAB_ROWS = 4_000_000 if FAST_DEV else 12_000_000

adm = pd.read_csv(
    MIMIC_HOSP / "admissions.csv.gz",
    usecols=["subject_id", "hadm_id", "admittime", "dischtime", "admission_type", "insurance", "marital_status", "race"],
    low_memory=False,
)
pat = pd.read_csv(
    MIMIC_HOSP / "patients.csv.gz",
    usecols=["subject_id", "gender", "anchor_age"],
    low_memory=False,
)
diag = pd.read_csv(
    MIMIC_HOSP / "diagnoses_icd.csv.gz",
    usecols=["hadm_id", "icd_code", "icd_version"],
    low_memory=False,
)

adm = adm.dropna(subset=["subject_id", "hadm_id"]).drop_duplicates(subset=["hadm_id"]) 
if len(adm) > N_ADMISSIONS:
    adm = adm.sample(N_ADMISSIONS, random_state=42)

cohort = adm.merge(pat, on="subject_id", how="left")
cohort["admittime"] = pd.to_datetime(cohort["admittime"], errors="coerce")
cohort["dischtime"] = pd.to_datetime(cohort["dischtime"], errors="coerce")
cohort["los_hours"] = (cohort["dischtime"] - cohort["admittime"]).dt.total_seconds() / 3600.0

print("Step2 cohort admissions:", cohort.shape)
print("FAST_DEV:", FAST_DEV, "| N_ADMISSIONS:", N_ADMISSIONS, "| MAX_LAB_ROWS:", MAX_LAB_ROWS)
print("Step2 config/load time (sec):", round(time.perf_counter() - _t_step2_cfg, 2))
cohort.head(3)

AssertionError: MIMIC hosp folder not found: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/mimic-iv-3.1/hosp

In [ ]:
# Step 2 label: CKD proxy from ICD diagnosis codes (admission-level)
_t_step2_label = time.perf_counter()

diag = diag.dropna(subset=["hadm_id", "icd_code", "icd_version"]).copy()
diag["icd_code"] = diag["icd_code"].astype(str).str.upper().str.strip()
diag["icd_version"] = pd.to_numeric(diag["icd_version"], errors="coerce")

is_ckd_icd9 = (diag["icd_version"] == 9) & diag["icd_code"].str.startswith("585")
is_ckd_icd10 = (diag["icd_version"] == 10) & diag["icd_code"].str.startswith("N18")
ckd_hadm = set(diag.loc[is_ckd_icd9 | is_ckd_icd10, "hadm_id"].astype("int64").tolist())

cohort["hadm_id"] = cohort["hadm_id"].astype("int64")
cohort["subject_id"] = cohort["subject_id"].astype("int64")
cohort["ckd_label"] = cohort["hadm_id"].isin(ckd_hadm).astype(int)

print("CKD positive rate (Step2):", round(float(cohort["ckd_label"].mean()), 4))
print("Step2 label time (sec):", round(time.perf_counter() - _t_step2_label, 2))
cohort[["hadm_id", "subject_id", "ckd_label"]].head(5)

CKD positive rate (Step2): 0.1507
Step2 label time (sec): 2.34


,hadm_id,subject_id,ckd_label
0,20755423,18521354,0
1,21482324,16559252,0
2,21276544,17237709,1
3,21268011,19397801,0
4,22494617,15463124,1


In [ ]:
# Step 2 lab feature extraction from labevents (chunked)
_t_step2_labs = time.perf_counter()

labitems = pd.read_csv(
    MIMIC_HOSP / "d_labitems.csv.gz",
    usecols=["itemid", "label"],
    low_memory=False,
)
labitems["label_l"] = labitems["label"].astype(str).str.lower().str.strip()

# Keep a compact, clinically meaningful set for first-pass tabular model
lab_keywords = {
    "creatinine": "lab_creatinine",
    "urea nitrogen": "lab_urea_nitrogen",
    "potassium": "lab_potassium",
    "sodium": "lab_sodium",
    "chloride": "lab_chloride",
    "bicarbonate": "lab_bicarbonate",
    "hemoglobin": "lab_hemoglobin",
    "platelet": "lab_platelet",
}

# Build itemid->feature map without row-wise iterrows loop.
itemid_to_feat = {}
for kw, feat in lab_keywords.items():
    hit = labitems["label_l"].str.contains(kw, regex=False, na=False)
    if hit.any():
        ids = labitems.loc[hit, "itemid"].astype("int64").tolist()
        for iid in ids:
            if iid not in itemid_to_feat:
                itemid_to_feat[iid] = feat

selected_itemids = np.array(sorted(itemid_to_feat.keys()), dtype=np.int64)
cohort_hadm = np.array(sorted(cohort["hadm_id"].astype("int64").unique()), dtype=np.int64)

acc = []
rows_seen = 0
chunks_seen = 0
for chunk in pd.read_csv(
    MIMIC_HOSP / "labevents.csv.gz",
    usecols=["hadm_id", "itemid", "valuenum"],
    chunksize=LAB_CHUNKSIZE,
    low_memory=False,
):
    chunks_seen += 1
    rows_seen += len(chunk)
    if MAX_LAB_ROWS is not None and rows_seen > MAX_LAB_ROWS:
        break

    x = chunk.dropna(subset=["hadm_id", "itemid", "valuenum"]).copy()
    if x.empty:
        continue

    # Apply the more selective filter first to shrink chunk quickly.
    x["itemid"] = x["itemid"].astype("int64")
    x = x[x["itemid"].isin(selected_itemids)]
    if x.empty:
        continue

    x["hadm_id"] = x["hadm_id"].astype("int64")
    x = x[x["hadm_id"].isin(cohort_hadm)]
    if x.empty:
        continue

    x["feature"] = x["itemid"].map(itemid_to_feat)
    g = x.groupby(["hadm_id", "feature"], as_index=False)["valuenum"].median()
    acc.append(g)

    # Periodically compact intermediate aggregates to control memory growth.
    if len(acc) >= 20:
        acc = [pd.concat(acc, ignore_index=True).groupby(["hadm_id", "feature"], as_index=False)["valuenum"].median()]

if acc:
    labs_long = pd.concat(acc, ignore_index=True)
    labs_long = labs_long.groupby(["hadm_id", "feature"], as_index=False)["valuenum"].median()
    labs = labs_long.pivot_table(index="hadm_id", columns="feature", values="valuenum", aggfunc="median")
    labs = labs.reset_index()
else:
    labs = pd.DataFrame({"hadm_id": cohort["hadm_id"].astype("int64")})

print("Chunks scanned from labevents:", chunks_seen)
print("Rows scanned from labevents:", rows_seen)
print("Lab feature frame:", labs.shape)
print("Step2 labs time (sec):", round(time.perf_counter() - _t_step2_labs, 2))
labs.head(3)

Chunks scanned from labevents: 3
Rows scanned from labevents: 6000000
Lab feature frame: (280, 9)
Step2 labs time (sec): 3.3


feature,hadm_id,lab_bicarbonate,lab_chloride,lab_creatinine,lab_hemoglobin,lab_platelet,lab_potassium,lab_sodium,lab_urea_nitrogen
0,20031141,24.0,103.0,1.4,7.75,194.5,5.10,137.0,39.0
1,20035879,21.5,105.0,4.5,8.80,205.0,3.85,137.0,41.0
2,20048401,24.0,108.0,0.6,10.40,214.0,4.20,143.0,9.0


In [ ]:
# Assemble Step2 model matrix + grouped split + train-only preprocessing
_t_step2_prep = time.perf_counter()

m2 = cohort.merge(labs, on="hadm_id", how="left")

# Categorical -> one-hot
cat_cols = ["gender", "admission_type", "insurance", "marital_status", "race"]
for c in cat_cols:
    if c not in m2.columns:
        m2[c] = "UNKNOWN"

base_num_cols = ["anchor_age", "los_hours"]
for c in base_num_cols:
    if c not in m2.columns:
        m2[c] = np.nan

lab_cols = [c for c in m2.columns if str(c).startswith("lab_")]
feature_df = m2[base_num_cols + cat_cols + lab_cols].copy()
feature_df = pd.get_dummies(feature_df, columns=cat_cols, dummy_na=True)

# Keep columns with at least minimal observed values
valid_cols = [c for c in feature_df.columns if feature_df[c].notna().sum() > 50]
X2_df = feature_df[valid_cols].replace([np.inf, -np.inf], np.nan)
y2 = m2["ckd_label"].astype(int).to_numpy()
groups2 = m2["subject_id"].astype("int64").to_numpy()

m2_tr, m2_va, m2_te = grouped_split_with_class_coverage(
    groups2,
    y2,
    test_size=0.20,
    val_size=0.20,
    random_state=42,
    max_tries=100,
)
assert_no_group_leakage(groups2, m2_tr, m2_va, m2_te)

X2_tr_df, X2_va_df, X2_te_df = X2_df.loc[m2_tr], X2_df.loc[m2_va], X2_df.loc[m2_te]
y2_tr, y2_va, y2_te = y2[m2_tr], y2[m2_va], y2[m2_te]

imp2 = SimpleImputer(strategy="median")
sc2 = StandardScaler()
X2_tr = sc2.fit_transform(imp2.fit_transform(X2_tr_df))
X2_va = sc2.transform(imp2.transform(X2_va_df))
X2_te = sc2.transform(imp2.transform(X2_te_df))

print("Step2 X shape:", X2_df.shape)
print("Step2 split sizes:", X2_tr.shape, X2_va.shape, X2_te.shape)
print("Step2 class balance train/val/test:", round(y2_tr.mean(), 3), round(y2_va.mean(), 3), round(y2_te.mean(), 3))
print("Step2 assemble/preprocess time (sec):", round(time.perf_counter() - _t_step2_prep, 2))

Step2 X shape: (15000, 68)
Step2 split sizes: (8993, 68) (3007, 68) (3000, 68)
Step2 class balance train/val/test: 0.148 0.148 0.161
Step2 assemble/preprocess time (sec): 0.07


In [ ]:
# Step2 baseline (Logistic Regression) + quick deep model (MLP)
_t_step2_models = time.perf_counter()

base2 = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)
base2.fit(X2_tr, y2_tr)

p2_va_base = base2.predict_proba(X2_va)[:, 1]
p2_te_base = base2.predict_proba(X2_te)[:, 1]

thr_grid2 = np.linspace(0.1, 0.9, 161)
best_thr2, _ = find_best_threshold(y2_va, p2_va_base, thr_grid2)
val2_base = enrich_binary_metrics(y2_va, p2_va_base, best_thr2)
test2_base = enrich_binary_metrics(y2_te, p2_te_base, best_thr2)

mlp2_epochs = 20 if FAST_DEV else 35
mlp2_result = train_tabular_mlp(
    X2_tr, y2_tr,
    X2_va, y2_va,
    hidden=(128, 64),
    dropout=0.20,
    epochs=mlp2_epochs,
    batch_size=64,
    lr=1e-3,
    weight_decay=1e-4,
    seed=42,
)

mlp2 = TabularMLP(n_features=X2_tr.shape[1], hidden=(128, 64), dropout=0.20, n_classes=2)
mlp2.load_state_dict(mlp2_result.model_state)
mlp2.eval()

with torch.no_grad():
    x2_va_t = torch.from_numpy(X2_va.astype(np.float32))
    x2_te_t = torch.from_numpy(X2_te.astype(np.float32))
    p2_va_mlp = softmax_rows(mlp2(x2_va_t).numpy())[:, 1]
    p2_te_mlp = softmax_rows(mlp2(x2_te_t).numpy())[:, 1]

best_thr2_mlp, _ = find_best_threshold(y2_va, p2_va_mlp, thr_grid2)
val2_mlp = enrich_binary_metrics(y2_va, p2_va_mlp, best_thr2_mlp)
test2_mlp = enrich_binary_metrics(y2_te, p2_te_mlp, best_thr2_mlp)

step2_summary = pd.DataFrame([
    {"model": "mimic_logreg", "accuracy": test2_base["accuracy"], "f1": test2_base["f1"], "auroc": test2_base["auroc"], "auprc": test2_base["auprc"], "ece": test2_base["ece"]},
    {"model": "mimic_tabular_mlp", "accuracy": test2_mlp["accuracy"], "f1": test2_mlp["f1"], "auroc": test2_mlp["auroc"], "auprc": test2_mlp["auprc"], "ece": test2_mlp["ece"]},
]).sort_values("auroc", ascending=False)

print("Step2 MLP epochs:", mlp2_epochs)
print("Step2 baseline AUROC:", round(test2_base["auroc"], 4))
print("Step2 MLP AUROC:", round(test2_mlp["auroc"], 4))
print("Step2 model/eval time (sec):", round(time.perf_counter() - _t_step2_models, 2))
step2_summary

Step2 MLP epochs: 20
Step2 baseline AUROC: 0.771
Step2 MLP AUROC: 0.7574
Step2 model/eval time (sec): 1.82


,model,accuracy,f1,auroc,auprc,ece
0,mimic_logreg,0.641333,0.418378,0.770998,0.368841,0.261138
1,mimic_tabular_mlp,0.633667,0.410091,0.757439,0.341929,0.023196


In [ ]:
# Save Step2 artifacts
_t_step2_save = time.perf_counter()

step2_artifact = {
    "cohort": "mimic_hosp_admission_level",
    "n_admissions_requested": int(N_ADMISSIONS),
    "rows_model": int(len(m2)), 
    "positive_rate": float(m2["ckd_label"].mean()),
    "n_features": int(X2_df.shape[1]),
    "lab_feature_cols": lab_cols,
    "baseline": {"val": val2_base, "test": test2_base, "threshold": float(best_thr2)},
    "tabular_mlp": {
        "val": val2_mlp,
        "test": test2_mlp,
        "threshold": float(best_thr2_mlp),
        "best_val_auc_training": float(mlp2_result.best_val_auc),
    },
}

step2_json = OUT / "step2_mimic_admission_branch.json"
step2_csv = OUT / "step2_mimic_admission_summary.csv"

with open(step2_json, "w", encoding="utf-8") as f:
    json.dump(step2_artifact, f, indent=2)
step2_summary.to_csv(step2_csv, index=False)

print("Saved Step2 artifacts:")
print("-", step2_json)
print("-", step2_csv)
print("Step2 save time (sec):", round(time.perf_counter() - _t_step2_save, 2))

Saved Step2 artifacts:
- /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_admission_branch.json
- /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_admission_summary.csv
Step2 save time (sec): 0.0


## 13) What to tell supervisor for Step 2

- Step 2 is a **within-MIMIC** admission-level branch, not cross-dataset patient merge.
- Label is CKD proxy from diagnosis coding (`585*` / `N18*`), with leakage-safe grouped split by `subject_id`.
- Features combine demographics/admission context + selected kidney-relevant labs from chunked `labevents` aggregation.
- We report baseline vs deep model under the same split and preprocessing policy.
- This branch becomes one tower in later multimodal fusion (Step 3+), alongside NHANES and wearable pathways.

## 14) Step 3: Multimodal Fusion Setup (NHANES + MIMIC)

Because NHANES and MIMIC do not share patient IDs, we start with a **late-fusion protocol** at model-output level:
- choose one champion model per branch,
- align branch outputs into a common risk score space,
- define fusion weights from validation quality,
- save a reproducible fusion specification for the next stage.

In [ ]:
# Step 3A: Load branch summaries and pick champion per branch

nhanes_csv = OUT / "from_scratch_clinical_nhanes_summary.csv"
mimic_csv = OUT / "step2_mimic_admission_summary.csv"

assert nhanes_csv.exists(), f"Missing NHANES summary: {nhanes_csv}"
assert mimic_csv.exists(), f"Missing MIMIC summary: {mimic_csv}"

nhanes_summary = pd.read_csv(nhanes_csv)
mimic_summary = pd.read_csv(mimic_csv)

# Normalize model names for branch tagging.
nhanes_summary = nhanes_summary.copy()
nhanes_summary["branch"] = "nhanes"
nhanes_summary["model"] = nhanes_summary["model"].astype(str)

mimic_summary = mimic_summary.copy()
mimic_summary["branch"] = "mimic"
mimic_summary["model"] = mimic_summary["model"].astype(str)

all_branch_models = pd.concat([
    nhanes_summary[["branch", "model", "accuracy", "f1", "auroc", "auprc", "ece"]],
    mimic_summary[["branch", "model", "accuracy", "f1", "auroc", "auprc", "ece"]],
], ignore_index=True)

# Branch champion criterion: prioritize AUROC, then AUPRC, then F1.
def pick_champion(df_branch):
    return df_branch.sort_values(["auroc", "auprc", "f1"], ascending=False).head(1)

champ_nhanes = pick_champion(all_branch_models[all_branch_models["branch"] == "nhanes"])
champ_mimic = pick_champion(all_branch_models[all_branch_models["branch"] == "mimic"])

fusion_champions = pd.concat([champ_nhanes, champ_mimic], ignore_index=True)
print("Branch champions selected:")
fusion_champions

Branch champions selected:


,branch,model,accuracy,f1,auroc,auprc,ece
0,nhanes,baseline_logreg,0.939400,0.691900,0.982200,0.851900,0.052700
1,mimic,mimic_logreg,0.641333,0.418378,0.770998,0.368841,0.261138


In [ ]:
# Step 3B: Build and save reproducible late-fusion specification

# Weight branch contribution by validation quality proxy (AUROC + AUPRC), then normalize.
fusion_spec = fusion_champions.copy()
fusion_spec["quality_score"] = fusion_spec["auroc"] + fusion_spec["auprc"]

q_sum = float(fusion_spec["quality_score"].sum())
if q_sum <= 0:
    fusion_spec["fusion_weight"] = 1.0 / max(len(fusion_spec), 1)
else:
    fusion_spec["fusion_weight"] = fusion_spec["quality_score"] / q_sum

fusion_spec = fusion_spec[[
    "branch", "model", "accuracy", "f1", "auroc", "auprc", "ece",
    "quality_score", "fusion_weight"
]].sort_values("branch").reset_index(drop=True)

# Define how to combine calibrated branch probabilities in Step 3 execution.
fusion_protocol = {
    "fusion_type": "late_fusion_probability_weighted",
    "probability_source": "calibrated_branch_probabilities",
    "weight_rule": "normalized(auroc + auprc)",
    "notes": [
        "NHANES and MIMIC are separate cohorts; no patient-level join is used.",
        "Branch probabilities must be calibrated before weighted fusion.",
        "Final threshold can be selected using Youden/F1/target-recall policy."
    ],
    "branches": [
        {
            "branch": str(r.branch),
            "model": str(r.model),
            "weight": float(r.fusion_weight),
            "quality_score": float(r.quality_score),
            "metrics": {
                "accuracy": float(r.accuracy),
                "f1": float(r.f1),
                "auroc": float(r.auroc),
                "auprc": float(r.auprc),
                "ece": float(r.ece),
            },
        }
        for r in fusion_spec.itertuples(index=False)
    ],
}

step3_csv = OUT / "step3_fusion_spec.csv"
step3_json = OUT / "step3_fusion_protocol.json"
fusion_spec.to_csv(step3_csv, index=False)
with open(step3_json, "w", encoding="utf-8") as f:
    json.dump(fusion_protocol, f, indent=2)

print("Saved Step 3 setup artifacts:")
print("-", step3_csv)
print("-", step3_json)
fusion_spec

Saved Step 3 setup artifacts:
- /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step3_fusion_spec.csv
- /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step3_fusion_protocol.json


,branch,model,accuracy,f1,auroc,auprc,ece,quality_score,fusion_weight
0,mimic,mimic_logreg,0.641333,0.418378,0.770998,0.368841,0.261138,1.139839,0.383276
1,nhanes,baseline_logreg,0.939400,0.691900,0.982200,0.851900,0.052700,1.834100,0.616724


## 14C) Step 3: Weighted Fusion Function + Threshold Policy

This cell operationalizes the late-fusion protocol:
- combine calibrated branch probabilities with learned fusion weights,
- choose decision threshold by policy (`youden`, `f1`, `target_recall`),
- return fused predictions and metrics.

Use this when branch-level calibrated probabilities are available for the same target cohort in future integration.

In [ ]:
# 14C) weighted fusion utilities
require_prereqs(
    "Section 14C",
    "binary_metrics_at_threshold", "enrich_binary_metrics",
    hint="Run ## 0) Setup and Imports before using fusion helpers.",
)
import numpy as np


def normalize_weights(weight_dict):
    keys = list(weight_dict.keys())
    vals = np.array([float(weight_dict[k]) for k in keys], dtype=float)
    s = vals.sum()
    if s <= 0:
        vals = np.ones_like(vals) / max(len(vals), 1)
    else:
        vals = vals / s
    return {k: float(v) for k, v in zip(keys, vals)}


def fuse_probabilities(prob_dict, weight_dict):
    """
    prob_dict: {'nhanes': np.ndarray, 'mimic': np.ndarray, ...}
    weight_dict: {'nhanes': float, 'mimic': float, ...}
    Returns weighted fused probability array.
    """
    w = normalize_weights(weight_dict)
    keys = [k for k in prob_dict.keys() if k in w]
    assert len(keys) > 0, "No overlapping branches between prob_dict and weight_dict"

    base_len = len(prob_dict[keys[0]])
    fused = np.zeros(base_len, dtype=float)
    for k in keys:
        p = np.asarray(prob_dict[k], dtype=float)
        assert len(p) == base_len, f"Length mismatch in branch '{k}'"
        fused += w[k] * p
    return np.clip(fused, 0.0, 1.0)


def pick_threshold_by_policy(y_true, p_pred, thr_grid, policy="youden", target_recall=0.85):
    best_thr = 0.5
    best_score = -1e18
    for t in thr_grid:
        m = binary_metrics_at_threshold(y_true, p_pred, threshold=float(t))
        if policy == "youden":
            score = m["sensitivity_recall"] + m["specificity"] - 1.0
        elif policy == "f1":
            score = m["f1"]
        elif policy == "target_recall":
            rec = m["sensitivity_recall"]
            spec = m["specificity"]
            score = spec if rec >= target_recall else (-1e6 + rec)
        else:
            raise ValueError(f"Unknown policy: {policy}")

        if score > best_score:
            best_score = score
            best_thr = float(t)
    return best_thr


def evaluate_fused_predictions(y_true, p_fused, threshold):
    m = enrich_binary_metrics(y_true, p_fused, threshold)
    return {
        "threshold": float(threshold),
        "accuracy": float(m["accuracy"]),
        "f1": float(m["f1"]),
        "recall": float(m["sensitivity_recall"]),
        "specificity": float(m["specificity"]),
        "auroc": float(m["auroc"]),
        "auprc": float(m["auprc"]),
        "ece": float(m["ece"]),
        "brier": float(m["brier"]),
    }


# Read branch weights from Step 3 spec for later reuse.
if "fusion_spec" in globals():
    fusion_weights = {r["branch"]: float(r["fusion_weight"]) for _, r in fusion_spec.iterrows()}
else:
    fusion_weights = {"nhanes": 0.616724, "mimic": 0.383276}

print("Fusion utilities ready.")
print("Default branch weights:", fusion_weights)
print("When aligned branch probabilities are available, call:")
print("  p_fused = fuse_probabilities({'nhanes': p_nh, 'mimic': p_mi}, fusion_weights)")
print("  thr = pick_threshold_by_policy(y_val, p_fused_val, np.linspace(0.1,0.9,161), policy='f1')")
print("  metrics = evaluate_fused_predictions(y_test, p_fused_test, thr)")

Fusion utilities ready.
Default branch weights: {'nhanes': 0.616724, 'mimic': 0.383276}
When aligned branch probabilities are available, call:
  p_fused = fuse_probabilities({'nhanes': p_nh, 'mimic': p_mi}, fusion_weights)
  thr = pick_threshold_by_policy(y_val, p_fused_val, np.linspace(0.1,0.9,161), policy='f1')
  metrics = evaluate_fused_predictions(y_test, p_fused_test, thr)


## 14D) Step 3 Fusion Demo (Branch-Level) — New Cell

Run **after `14A` + `14C`**.

Demonstrates weighted late fusion using **branch validation AUROC** as proxy branch scores on a synthetic aligned cohort (protocol demo until wearable branch is integrated).

In [ ]:
# Step 3D: fusion demo (branch-quality-informed proxy)
nhanes_csv = OUT / "from_scratch_clinical_nhanes_summary.csv"
mimic_csv = OUT / "step2_mimic_admission_summary.csv"
nh = pd.read_csv(nhanes_csv)
mi = pd.read_csv(mimic_csv)
p_nh_level = float(nh.sort_values("auroc", ascending=False).iloc[0]["auroc"])
p_mi_level = float(mi.sort_values("auroc", ascending=False).iloc[0]["auroc"])
n_demo = 500
rng = np.random.default_rng(7)
y_demo = rng.binomial(1, p=0.15, size=n_demo)
p_nh_demo = np.clip(rng.normal(p_nh_level, 0.08, size=n_demo), 0, 1)
p_mi_demo = np.clip(rng.normal(p_mi_level, 0.10, size=n_demo), 0, 1)
p_fused_demo = fuse_probabilities({"nhanes": p_nh_demo, "mimic": p_mi_demo}, fusion_weights)
thr_demo = pick_threshold_by_policy(y_demo, p_fused_demo, np.linspace(0.1, 0.9, 161), policy="f1")
metrics_demo = evaluate_fused_predictions(y_demo, p_fused_demo, thr_demo)
fusion_demo_summary = pd.DataFrame([
    {"component": "nhanes_proxy_level", "auroc": float(safe_auroc(y_demo, p_nh_demo)), "auprc": float(safe_auprc(y_demo, p_nh_demo)), "proxy_center": p_nh_level},
    {"component": "mimic_proxy_level", "auroc": float(safe_auroc(y_demo, p_mi_demo)), "auprc": float(safe_auprc(y_demo, p_mi_demo)), "proxy_center": p_mi_level},
    {"component": "fused_weighted", "threshold": metrics_demo["threshold"], "accuracy": metrics_demo["accuracy"], "f1": metrics_demo["f1"], "recall": metrics_demo["recall"], "specificity": metrics_demo["specificity"], "auroc": metrics_demo["auroc"], "auprc": metrics_demo["auprc"], "ece": metrics_demo["ece"]},
])
print("Fusion weights:", fusion_weights)
print("NOTE: replace with aligned real branch probabilities after wearable integration.")
fusion_demo_summary
fusion_demo_csv = OUT / "step3_fusion_demo_summary.csv"
fusion_demo_summary.to_csv(fusion_demo_csv, index=False)
print("Saved:", fusion_demo_csv)


Fusion weights: {'nhanes': 0.616724, 'mimic': 0.383276}
NOTE: replace with aligned real branch probabilities after wearable integration.
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step3_fusion_demo_summary.csv


## Viz-1) Calibration Reliability Plot (MIMIC)

Run **after `12B-Resume`** and **`12E`** (or after `15B`).

In [ ]:
# Viz-1) Calibration reliability — MIMIC test set
import pickle
import time
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

if "OUT" not in globals():
    OUT = Path.cwd() / "outputs" / "supervisor_runs"
if "CHECKPOINT_PATH" not in globals():
    CHECKPOINT_PATH = OUT / "step2_mimic_checkpoint.pkl"

_t_v1 = time.perf_counter()

if "X2_te" not in globals() and CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "rb") as _f:
        _ck = pickle.load(_f)
    for _k, _v in _ck.items():
        if _k != "step2_summary":
            globals()[_k] = _v
    print("Loaded checkpoint for Viz-1")

assert "X2_te" in globals(), "Run 12B or 12B-Resume first."

if "cal_sigmoid" not in globals():
    cal_sigmoid = CalibratedClassifierCV(FrozenEstimator(base2), method="sigmoid")
    cal_sigmoid.fit(X2_va, y2_va)

p_uncal = base2.predict_proba(X2_te)[:, 1]
p_cal = cal_sigmoid.predict_proba(X2_te)[:, 1]

fig, ax = plt.subplots(figsize=(6, 5))
for label, p, color in [
    ("Uncalibrated LogReg", p_uncal, "#d62728"),
    ("Sigmoid calibrated", p_cal, "#2ca02c"),
]:
    frac_pos, mean_pred = calibration_curve(y2_te, p, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", label=label, color=color, linewidth=2)

ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration", alpha=0.6)
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("MIMIC test — calibration reliability")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
fig.tight_layout()

out_v1 = OUT / "fig_mimic_calibration_reliability.png"
fig.savefig(out_v1, dpi=150)
plt.close(fig)
print("Saved:", out_v1)
print("Viz-1 time (sec):", round(time.perf_counter() - _t_v1, 2))


Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/fig_mimic_calibration_reliability.png
Viz-1 time (sec): 0.33


## Viz-2) MIMIC Model AUROC Comparison

Run **after `12F`**.

In [ ]:
# Viz-2) MIMIC model AUROC comparison (test set)
import time
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

if "OUT" not in globals():
    OUT = Path.cwd() / "outputs" / "supervisor_runs"

_t_v2 = time.perf_counter()

ext_csv = OUT / "step2_mimic_admission_summary_extended.csv"
assert ext_csv.exists(), f"Run 12F first: {ext_csv}"
df_v2 = pd.read_csv(ext_csv).sort_values("auroc", ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#1f77b4"] * len(df_v2)
ax.barh(df_v2["model"], df_v2["auroc"], color=colors)
ax.set_xlim(0.72, 0.80)
ax.set_xlabel("Test AUROC")
ax.set_title("MIMIC admission branch — model comparison")
for _, r in df_v2.iterrows():
    ax.text(r["auroc"] + 0.001, r["model"], f"{r['auroc']:.3f}", va="center", fontsize=9)
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()

out_v2 = OUT / "fig_mimic_model_auroc.png"
fig.savefig(out_v2, dpi=150)
plt.close(fig)
print("Saved:", out_v2)
print("Viz-2 time (sec):", round(time.perf_counter() - _t_v2, 2))


Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/fig_mimic_model_auroc.png
Viz-2 time (sec): 0.15


## Viz-3) MIMIC SHAP Top Features Bar Chart

Run **after `15B`**.

In [ ]:
# Viz-3) SHAP top-10 features bar chart
import time
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

if "OUT" not in globals():
    OUT = Path.cwd() / "outputs" / "supervisor_runs"

_t_v3 = time.perf_counter()

shap_csv = OUT / "step2_mimic_shap_top15.csv"
assert shap_csv.exists(), f"Run 15B first: {shap_csv}"
df_v3 = pd.read_csv(shap_csv).head(10).sort_values("mean_abs_shap", ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(df_v3["feature"], df_v3["mean_abs_shap"], color="#9467bd")
ax.set_xlabel("Mean |SHAP|")
ax.set_title("MIMIC logreg_sigmoid_cal — top 10 features (SHAP)")
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()

out_v3 = OUT / "fig_mimic_shap_top10.png"
fig.savefig(out_v3, dpi=150)
plt.close(fig)
print("Saved:", out_v3)
print("Viz-3 time (sec):", round(time.perf_counter() - _t_v3, 2))


## 16) Wearable Branch (WESAD) — Demo Baseline

Run **after setup (`## 0`)** and before `14D-final`.

Minimal viable branch: load WESAD wrist streams via `resolve_wesad_root(ROOT)`, window-level stats, stress-vs-baseline proxy labels (not CKD), logistic/RF baseline, export branch probabilities for fusion stub.

**Limitations:** no CKD labels in WESAD; no patient-level join with NHANES/MIMIC; demo-grade features only.

In [ ]:
# 16) WESAD wearable branch (demo-grade baseline)
require_prereqs(
    "Section 16 (WESAD)",
    "ROOT", "OUT", "resolve_wesad_root", "enrich_binary_metrics", "pd", "np", "pickle", "json",
    hint="Run kernel check, then ## 0) Setup and Imports (and DATA PREFLIGHT) before this cell.",
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

WESAD_ROOT = resolve_wesad_root(ROOT)
assert WESAD_ROOT.exists(), f"WESAD not found: {WESAD_ROOT}"

WEARABLE_WINDOW_SAMPLES = 1920  # ~30s at 64 Hz wrist BVP
WEARABLE_STRIDE = 960
WEARABLE_MAX_WINDOWS_PER_SUBJECT = 120


def _wesad_window_stats(x):
    x = np.asarray(x, dtype=float).reshape(-1)
    if x.size == 0:
        return [np.nan, np.nan, np.nan, np.nan]
    return [float(np.nanmean(x)), float(np.nanstd(x)), float(np.nanmin(x)), float(np.nanmax(x))]


def _load_wesad_subject_pkl(subject_dir: Path):
    sid = subject_dir.name
    pkl_path = subject_dir / f"{sid}.pkl"
    with open(pkl_path, "rb") as f:
        return pickle.load(f, encoding="latin1")


def extract_wesad_wrist_windows(subject_dict):
    """Window-level wrist features; binary label stress(2) vs baseline(1)."""
    wrist = subject_dict["signal"]["wrist"]
    label = np.asarray(subject_dict["label"]).reshape(-1)
    sid = str(subject_dict.get("subject", "unknown"))

    bvp = np.asarray(wrist["BVP"], dtype=float).reshape(-1)
    eda = np.asarray(wrist["EDA"], dtype=float).reshape(-1)
    temp = np.asarray(wrist["TEMP"], dtype=float).reshape(-1)
    acc = np.asarray(wrist["ACC"], dtype=float)

    n = len(bvp)
    rows = []
    starts = list(range(0, max(n - WEARABLE_WINDOW_SAMPLES, 0) + 1, WEARABLE_STRIDE))
    if len(starts) > WEARABLE_MAX_WINDOWS_PER_SUBJECT:
        pick = np.linspace(0, len(starts) - 1, WEARABLE_MAX_WINDOWS_PER_SUBJECT, dtype=int)
        starts = [starts[i] for i in pick]

    for start in starts:
        end = start + WEARABLE_WINDOW_SAMPLES
        if end > n:
            continue

        l0 = int(start * len(label) / n)
        l1 = int(end * len(label) / n)
        l1 = min(max(l1, l0 + 1), len(label))
        seg = label[l0:l1]
        vals, counts = np.unique(seg, return_counts=True)
        maj = int(vals[np.argmax(counts)])
        if maj not in (1, 2):
            continue
        y = 1 if maj == 2 else 0

        acc_win = acc[start:end]
        acc_mag = np.linalg.norm(acc_win, axis=1) if acc_win.ndim == 2 else acc_win.reshape(-1)

        feat_pack = {}
        for ch_name, arr in [
            ("bvp", bvp[start:end]),
            ("eda", eda[min(start, len(eda) - 1) : min(end, len(eda))]),
            ("temp", temp[min(start, len(temp) - 1) : min(end, len(temp))]),
            ("acc_mag", acc_mag),
        ]:
            stats = _wesad_window_stats(arr)
            for stat_name, val in zip(("mean", "std", "min", "max"), stats):
                feat_pack[f"{ch_name}_{stat_name}"] = val

        rows.append({"subject_id": sid, "window_start": int(start), "y_stress": y, **feat_pack})

    return rows


subject_dirs = sorted([p.parent for p in WESAD_ROOT.glob("S*/S*.pkl")])
print("WESAD subjects:", len(subject_dirs), "| root:", WESAD_ROOT)

wearable_rows = []
for sd in subject_dirs:
    wearable_rows.extend(extract_wesad_wrist_windows(_load_wesad_subject_pkl(sd)))

wearable_df = pd.DataFrame(wearable_rows)
feature_cols = [c for c in wearable_df.columns if c not in ("subject_id", "window_start", "y_stress")]
X = wearable_df[feature_cols].to_numpy(dtype=float)
y = wearable_df["y_stress"].to_numpy(dtype=int)
groups = wearable_df["subject_id"].to_numpy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

model_candidates = [
    (
        "wearable_logreg",
        Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler()),
                ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
            ]
        ),
    ),
    (
        "wearable_rf",
        Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                (
                    "clf",
                    RandomForestClassifier(
                        n_estimators=120,
                        random_state=7,
                        class_weight="balanced_subsample",
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
    ),
]

best_name, best_pipe, best_metrics = None, None, None
for model_name, pipe in model_candidates:
    pipe.fit(X[train_idx], y[train_idx])
    p_test = pipe.predict_proba(X[test_idx])[:, 1]
    m = enrich_binary_metrics(y[test_idx], p_test, threshold=0.5)
    print(f"{model_name}: AUROC={m['auroc']:.4f} AUPRC={m['auprc']:.4f} F1={m['f1']:.4f}")
    if best_metrics is None or m["auroc"] > best_metrics["auroc"]:
        best_name, best_pipe, best_metrics = model_name, pipe, m

best_pipe.fit(X, y)
p_all = best_pipe.predict_proba(X)[:, 1]

wearable_probs = wearable_df[["subject_id", "window_start", "y_stress"]].copy()
wearable_probs["p_stress"] = p_all
wearable_probs["split"] = np.where(np.isin(np.arange(len(wearable_df)), test_idx), "test", "train")

wearable_probs_csv = OUT / "wearable_branch_probs_demo.csv"
wearable_probs.to_csv(wearable_probs_csv, index=False)

wearable_summary = {
    "branch": "wearable",
    "dataset": "WESAD",
    "wesad_root": str(WESAD_ROOT),
    "task": "binary_stress_vs_baseline_proxy",
    "label_map": {"1": "baseline", "2": "stress"},
    "model": best_name,
    "n_subjects": int(wearable_df["subject_id"].nunique()),
    "n_windows": int(len(wearable_df)),
    "feature_source": "wrist_BVP_EDA_TEMP_ACC_window_stats",
    "window_samples": WEARABLE_WINDOW_SAMPLES,
    "stride": WEARABLE_STRIDE,
    "max_windows_per_subject": WEARABLE_MAX_WINDOWS_PER_SUBJECT,
    "limitations": [
        "No CKD labels in WESAD; stress detection is a physiology proxy only.",
        "No patient-level alignment with NHANES/MIMIC cohorts.",
        "Demo-grade window aggregation; not a production wearable encoder.",
    ],
    "metrics_holdout_group_split": {
        k: float(best_metrics[k])
        for k in ("auroc", "auprc", "f1", "accuracy", "sensitivity_recall", "specificity", "ece", "brier")
    },
}

wearable_summary_json = OUT / "wearable_branch_summary.json"
with open(wearable_summary_json, "w", encoding="utf-8") as f:
    json.dump(wearable_summary, f, indent=2)

WEARABLE_STATUS = {
    "dataset_path_expected": str(WESAD_ROOT),
    "integrated": True,
    "model": best_name,
    "artifact_summary": str(wearable_summary_json),
    "artifact_probs": str(wearable_probs_csv),
    "notes": "Demo wearable branch trained; CKD alignment still pending.",
}
with open(OUT / "wearable_branch_status.json", "w", encoding="utf-8") as f:
    json.dump(WEARABLE_STATUS, f, indent=2)

print("Champion wearable model:", best_name)
print("Saved:", wearable_summary_json)
print("Saved:", wearable_probs_csv)
print("Saved:", OUT / "wearable_branch_status.json")
wearable_summary


WESAD subjects: 15 | root: /Users/md.shadmantahsin/Desktop/STUDY/Dataset/WESAD
wearable_logreg: AUROC=0.6690 AUPRC=0.5867 F1=0.4918
wearable_rf: AUROC=0.7639 AUPRC=0.7747 F1=0.6667
Champion wearable model: wearable_rf
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/wearable_branch_summary.json
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/wearable_branch_probs_demo.csv
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/wearable_branch_status.json


{'branch': 'wearable',
 'dataset': 'WESAD',
 'wesad_root': '/Users/md.shadmantahsin/Desktop/STUDY/Dataset/WESAD',
 'task': 'binary_stress_vs_baseline_proxy',
 'label_map': {'1': 'baseline', '2': 'stress'},
 'model': 'wearable_rf',
 'n_subjects': 15,
 'n_windows': 578,
 'feature_source': 'wrist_BVP_EDA_TEMP_ACC_window_stats',
 'window_samples': 1920,
 'stride': 960,
 'max_windows_per_subject': 120,
 'limitations': ['No CKD labels in WESAD; stress detection is a physiology proxy only.',
  'No patient-level alignment with NHANES/MIMIC cohorts.',
  'Demo-grade window aggregation; not a production wearable encoder.'],
 'metrics_holdout_group_split': {'auroc': 0.7638561320754716,
  'auprc': 0.7746987730406824,
  'f1': 0.6666666666666666,
  'accuracy': 0.8120805369127517,
  'sensitivity_recall': 0.5283018867924528,
  'specificity': 0.96875,
  'ece': 0.12337807606263983,
  'brier': 0.15180648769574942}}

## 14D-final) Step 3 Fusion Stub (NHANES + MIMIC + Wearable)

Run **after `14A` + `14C` + `16`**.

Uses exported wearable branch quality to extend the Step 3 fusion demo from 2-branch proxy (`14D`) to a **3-branch protocol stub**. Still **not** patient-aligned across cohorts; documents the path from proxy fusion to final evaluation when aligned probabilities exist.

In [ ]:
# 14D-final: 3-branch fusion stub (proxy cohort; wearable probs integrated at branch level)
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Bootstrap paths when ## 0) Setup was not run in this kernel session
ROOT = globals().get("ROOT", Path.cwd())
OUT = globals().get("OUT", ROOT / "outputs" / "supervisor_runs")
OUT.mkdir(parents=True, exist_ok=True)

_missing = [n for n in ("fuse_probabilities", "pick_threshold_by_policy", "evaluate_fused_predictions", "safe_auroc") if n not in globals()]
if _missing:
    raise RuntimeError(
        f"14D-final missing: {_missing}. Run ## 0) Setup and Imports, then 14C (fusion utilities). "
        f"Artifacts dir: {OUT}"
    )

assert (OUT / "wearable_branch_summary.json").exists(), "Run section 16 first."
assert (OUT / "step3_fusion_spec.csv").exists(), "Run 14A/14B first."

with open(OUT / "wearable_branch_summary.json", "r", encoding="utf-8") as f:
    wearable_pack = json.load(f)

fusion_spec_base = pd.read_csv(OUT / "step3_fusion_spec.csv")
wearable_metrics = wearable_pack["metrics_holdout_group_split"]

wearable_row = pd.DataFrame(
    [
        {
            "branch": "wearable",
            "model": wearable_pack["model"],
            "accuracy": wearable_metrics.get("accuracy", np.nan),
            "f1": wearable_metrics.get("f1", np.nan),
            "auroc": wearable_metrics.get("auroc", np.nan),
            "auprc": wearable_metrics.get("auprc", np.nan),
            "ece": wearable_metrics.get("ece", np.nan),
        }
    ]
)

fusion_spec_3 = pd.concat([fusion_spec_base, wearable_row], ignore_index=True)
fusion_spec_3["quality_score"] = fusion_spec_3["auroc"] + fusion_spec_3["auprc"]
q_sum = float(fusion_spec_3["quality_score"].sum())
fusion_spec_3["fusion_weight"] = fusion_spec_3["quality_score"] / q_sum if q_sum > 0 else 1.0 / len(fusion_spec_3)

fusion_weights_3 = {r["branch"]: float(r["fusion_weight"]) for _, r in fusion_spec_3.iterrows()}

nhanes_csv = OUT / "from_scratch_clinical_nhanes_summary.csv"
mimic_csv = OUT / "step2_mimic_admission_summary.csv"
nh = pd.read_csv(nhanes_csv)
mi = pd.read_csv(mimic_csv)
p_nh_level = float(nh.sort_values("auroc", ascending=False).iloc[0]["auroc"])
p_mi_level = float(mi.sort_values("auroc", ascending=False).iloc[0]["auroc"])
p_wear_level = float(wearable_metrics["auroc"])

n_demo = 500
rng = np.random.default_rng(11)
y_demo = rng.binomial(1, p=0.15, size=n_demo)
p_nh_demo = np.clip(rng.normal(p_nh_level, 0.08, size=n_demo), 0, 1)
p_mi_demo = np.clip(rng.normal(p_mi_level, 0.10, size=n_demo), 0, 1)
p_wear_demo = np.clip(rng.normal(p_wear_level, 0.12, size=n_demo), 0, 1)

p_fused_3 = fuse_probabilities(
    {"nhanes": p_nh_demo, "mimic": p_mi_demo, "wearable": p_wear_demo},
    fusion_weights_3,
)
thr_3 = pick_threshold_by_policy(y_demo, p_fused_3, np.linspace(0.1, 0.9, 161), policy="f1")
metrics_3 = evaluate_fused_predictions(y_demo, p_fused_3, thr_3)

fusion_final_stub = pd.DataFrame(
    [
        {"component": "nhanes_proxy_level", "auroc": float(safe_auroc(y_demo, p_nh_demo)), "proxy_center": p_nh_level},
        {"component": "mimic_proxy_level", "auroc": float(safe_auroc(y_demo, p_mi_demo)), "proxy_center": p_mi_level},
        {"component": "wearable_branch_level", "auroc": float(safe_auroc(y_demo, p_wear_demo)), "proxy_center": p_wear_level},
        {
            "component": "fused_weighted_3branch",
            "threshold": metrics_3["threshold"],
            "accuracy": metrics_3["accuracy"],
            "f1": metrics_3["f1"],
            "recall": metrics_3["recall"],
            "specificity": metrics_3["specificity"],
            "auroc": metrics_3["auroc"],
            "auprc": metrics_3["auprc"],
            "ece": metrics_3["ece"],
        },
    ]
)

fusion_final_csv = OUT / "step3_fusion_final_stub_summary.csv"
fusion_spec_3_csv = OUT / "step3_fusion_spec_3branch.csv"
fusion_final_stub.to_csv(fusion_final_csv, index=False)
fusion_spec_3.to_csv(fusion_spec_3_csv, index=False)

fusion_final_status = {
    "status": "wearable_branch_ready_proxy_fusion_pending_alignment",
    "fusion_demo_file": "step3_fusion_demo_summary.csv",
    "fusion_final_stub_file": "step3_fusion_final_stub_summary.csv",
    "fusion_spec_3branch_file": "step3_fusion_spec_3branch.csv",
    "protocol_file": "step3_fusion_protocol.json",
    "wearable_summary_file": "wearable_branch_summary.json",
    "wearable_probs_file": "wearable_branch_probs_demo.csv",
    "blocker": "No aligned patient-level NHANES+MIMIC+wearable cohort; this cell is a protocol stub only.",
    "dataset_root_resolved": str(_study_dataset_root(ROOT)) if "_study_dataset_root" in globals() else str(ROOT.parent.parent / "Dataset"),
    "wesad_path": str(resolve_wesad_root(ROOT)) if "resolve_wesad_root" in globals() else str(ROOT.parent.parent / "Dataset" / "WESAD"),
    "fusion_weights_3branch": fusion_weights_3,
    "next_steps": [
        "Replace proxy centers with aligned branch probabilities on a shared validation design",
        "Calibrate wearable probabilities before production fusion",
        "Optional: compare static weighted fusion vs meta-learner",
    ],
}
with open(OUT / "fusion_final_evaluation_status.json", "w", encoding="utf-8") as f:
    json.dump(fusion_final_status, f, indent=2)

print("3-branch fusion weights:", fusion_weights_3)
print("NOTE: proxy demo only — not patient-aligned across NHANES/MIMIC/WESAD.")
print("Saved:", fusion_final_csv)
print("Saved:", fusion_spec_3_csv)
print("Saved:", OUT / "fusion_final_evaluation_status.json")
fusion_final_stub


3-branch fusion weights: {'mimic': 0.254606014326594, 'nhanes': 0.40535635799848013, 'wearable': 0.3400376276749258}
NOTE: proxy demo only — not patient-aligned across NHANES/MIMIC/WESAD.
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step3_fusion_final_stub_summary.csv
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step3_fusion_spec_3branch.csv
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/fusion_final_evaluation_status.json


,component,auroc,proxy_center,threshold,accuracy,f1,recall,specificity,auprc,ece
0,nhanes_proxy_level,0.474983,0.992200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mimic_proxy_level,0.504613,0.766264,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,wearable_branch_level,0.523950,0.763856,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,fused_weighted_3branch,0.506911,NaN,0.775,0.196,0.252788,0.957746,0.06993,0.155926,0.701997


## 14E) Meta-Learner Fusion vs Static Weights (Freeze #5)

In [ ]:
# 14E: meta-learner fusion on branch probability features (no patient join)
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as MetaLogReg

ROOT = globals().get("ROOT", Path.cwd())
OUT = globals().get("OUT", ROOT / "outputs" / "supervisor_runs")
OUT.mkdir(parents=True, exist_ok=True)

if "fuse_probabilities" not in globals():
    from src.eval.metrics import binary_metrics_at_threshold, safe_auroc, safe_auprc
    from src.eval.calibration import expected_calibration_error, brier_score_binary

    def enrich_binary_metrics(y_true, p_pred, threshold):
        pack = binary_metrics_at_threshold(y_true, p_pred, threshold=float(threshold))
        pack["auroc"] = float(safe_auroc(y_true, p_pred))
        pack["auprc"] = float(safe_auprc(y_true, p_pred))
        pack["ece"] = float(expected_calibration_error(y_true, p_pred))
        pack["brier"] = float(brier_score_binary(y_true, p_pred))
        return pack

    def normalize_weights(weight_dict):
        keys = list(weight_dict.keys())
        vals = np.array([float(weight_dict[k]) for k in keys], dtype=float)
        s = vals.sum()
        if s <= 0:
            vals = np.ones_like(vals) / max(len(vals), 1)
        else:
            vals = vals / s
        return {k: float(v) for k, v in zip(keys, vals)}

    def fuse_probabilities(prob_dict, weight_dict):
        w = normalize_weights(weight_dict)
        keys = [k for k in prob_dict.keys() if k in w]
        assert len(keys) > 0, "No overlapping branches between prob_dict and weight_dict"
        base_len = len(prob_dict[keys[0]])
        fused = np.zeros(base_len, dtype=float)
        for k in keys:
            p = np.asarray(prob_dict[k], dtype=float)
            assert len(p) == base_len, f"Length mismatch in branch '{k}'"
            fused += w[k] * p
        return np.clip(fused, 0.0, 1.0)

    def pick_threshold_by_policy(y_true, p_pred, thr_grid, policy="youden", target_recall=0.85):
        best_thr = 0.5
        best_score = -1e18
        for t in thr_grid:
            m = binary_metrics_at_threshold(y_true, p_pred, threshold=float(t))
            if policy == "youden":
                score = m["sensitivity_recall"] + m["specificity"] - 1.0
            elif policy == "f1":
                score = m["f1"]
            elif policy == "target_recall":
                rec = m["sensitivity_recall"]
                spec = m["specificity"]
                score = spec if rec >= target_recall else (-1e6 + rec)
            else:
                raise ValueError(f"Unknown policy: {policy}")
            if score > best_score:
                best_score = score
                best_thr = float(t)
        return best_thr

    def evaluate_fused_predictions(y_true, p_fused, threshold):
        m = enrich_binary_metrics(y_true, p_fused, threshold)
        return {
            "threshold": float(threshold),
            "accuracy": float(m["accuracy"]),
            "f1": float(m["f1"]),
            "recall": float(m["sensitivity_recall"]),
            "specificity": float(m["specificity"]),
            "auroc": float(m["auroc"]),
            "auprc": float(m["auprc"]),
            "ece": float(m["ece"]),
            "brier": float(m["brier"]),
        }

assert (OUT / "step3_fusion_spec.csv").exists(), "Run 14A/14B first."

fusion_spec_meta = pd.read_csv(OUT / "step3_fusion_spec.csv")
fusion_weights_meta = {r["branch"]: float(r["fusion_weight"]) for _, r in fusion_spec_meta.iterrows()}

nhanes_csv = OUT / "from_scratch_clinical_nhanes_summary.csv"
mimic_csv = OUT / "step2_mimic_admission_summary.csv"
nh = pd.read_csv(nhanes_csv)
mi = pd.read_csv(mimic_csv)
p_nh_level = float(nh.sort_values("auroc", ascending=False).iloc[0]["auroc"])
p_mi_level = float(mi.sort_values("auroc", ascending=False).iloc[0]["auroc"])

# Synthetic branch-level holdout/demo protocol (strict: no cross-cohort patient join)
n_hold = 1200
rng_meta = np.random.default_rng(17)
y_hold = rng_meta.binomial(1, p=0.15, size=n_hold)
p_nh_hold = np.clip(rng_meta.normal(p_nh_level, 0.08, size=n_hold), 0, 1)
p_mi_hold = np.clip(rng_meta.normal(p_mi_level, 0.10, size=n_hold), 0, 1)

X_meta = np.column_stack([p_nh_hold, p_mi_hold])
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
    X_meta, y_hold, test_size=0.30, random_state=17, stratify=y_hold
)

meta_clf = MetaLogReg(max_iter=2000, class_weight="balanced", random_state=17)
meta_clf.fit(X_tr_m, y_tr_m)
p_meta_te = meta_clf.predict_proba(X_te_m)[:, 1]

thr_grid_meta = np.linspace(0.1, 0.9, 161)
p_nh_te = X_te_m[:, 0]
p_mi_te = X_te_m[:, 1]
p_static_te = fuse_probabilities({"nhanes": p_nh_te, "mimic": p_mi_te}, fusion_weights_meta)
thr_static = pick_threshold_by_policy(y_te_m, p_static_te, thr_grid_meta, policy="f1")
thr_meta = pick_threshold_by_policy(y_te_m, p_meta_te, thr_grid_meta, policy="f1")

m_static = evaluate_fused_predictions(y_te_m, p_static_te, thr_static)
m_meta = evaluate_fused_predictions(y_te_m, p_meta_te, thr_meta)

fusion_meta_comparison = pd.DataFrame([
    {"fusion_method": "static_weighted", **m_static},
    {"fusion_method": "meta_logreg_branch_probs", **m_meta},
])
meta_csv = OUT / "fusion_meta_comparison.csv"
fusion_meta_comparison.to_csv(meta_csv, index=False)
print("Saved:", meta_csv)
fusion_meta_comparison

winner = fusion_meta_comparison.sort_values(["auroc", "f1"], ascending=False).iloc[0]
fusion_status = {
    "status": "meta_fusion_comparison_complete_proxy_holdout",
    "fusion_demo_file": "step3_fusion_demo_summary.csv",
    "fusion_final_stub_file": "step3_fusion_final_stub_summary.csv",
    "fusion_meta_comparison_file": "fusion_meta_comparison.csv",
    "protocol_file": "step3_fusion_protocol.json",
    "wearable_summary_file": "wearable_branch_summary.json",
    "wearable_probs_file": "wearable_branch_probs_demo.csv",
    "blocker": "No aligned patient-level NHANES+MIMIC+wearable cohort; comparisons use branch-level proxy holdout only.",
    "dataset_root_resolved": str(_study_dataset_root(ROOT)) if "_study_dataset_root" in globals() else str(ROOT.parent.parent / "Dataset"),
    "wesad_path": str(resolve_wesad_root(ROOT)) if "resolve_wesad_root" in globals() else str(ROOT.parent.parent / "Dataset" / "WESAD"),
    "freeze_5_recommendation": str(winner["fusion_method"]),
    "next_steps": [
        "Document Freeze #5 choice in thesis (static vs meta on proxy holdout).",
        "Replace proxy fusion with aligned evaluation when a shared validation design exists.",
    ],
}
with open(OUT / "fusion_final_evaluation_status.json", "w", encoding="utf-8") as f:
    json.dump(fusion_status, f, indent=2)
print("Updated:", OUT / "fusion_final_evaluation_status.json")


Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/fusion_meta_comparison.csv
Updated: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/fusion_final_evaluation_status.json


In [ ]:
nh = pd.read_csv(OUT / "from_scratch_clinical_nhanes_summary.csv") if (OUT / "from_scratch_clinical_nhanes_summary.csv").exists() else pd.DataFrame()

# Freeze sign-off registry (Phases 1–5)
freeze_signoff = {
    "freeze_1_branch_champions": {
        "nhanes": str(nh.sort_values("auroc", ascending=False).iloc[0]["model"]) if (OUT / "from_scratch_clinical_nhanes_summary.csv").exists() else None,
        "mimic": "logreg_sigmoid_cal",
        "wearable": "wearable_rf_proxy_stress",
        "artifacts": [
            "from_scratch_clinical_nhanes_summary_extended.csv",
            "step2_mimic_admission_summary_extended.csv",
        ],
    },
    "freeze_2_threshold_calibration": {
        "primary_policy": "f1",
        "screening_policy": "target_recall",
        "mimic_model": "logreg_sigmoid_cal",
        "artifact": "final_reporting_lock.json",
    },
    "freeze_3_robustness": {
        "n_seeds": 5,
        "bootstrap_ci_metrics": ["auroc", "f1"],
        "artifacts": [
            "step2_mimic_robustness_per_seed.csv",
            "step2_mimic_robustness_mean_std.csv",
            "step2_mimic_robustness_ci95.csv",
        ],
    },
    "freeze_4_interpretability": {
        "global": "step2_mimic_permutation_importance_top15.csv",
        "case_examples": "step2_mimic_case_examples.csv",
        "shap_status": "step2_mimic_shap_status.json",
    },
    "freeze_5_fusion": {
        "static_spec": "step3_fusion_spec.csv",
        "meta_comparison": "fusion_meta_comparison.csv",
        "recommendation_file": "fusion_final_evaluation_status.json",
    },
}
with open(OUT / "freeze_signoff.json", "w", encoding="utf-8") as f:
    json.dump(freeze_signoff, f, indent=2)
print("Saved:", OUT / "freeze_signoff.json")


## 17) Progress Report Artifact Pack (always runnable)

Builds a one-page supervisor summary from saved artifacts in `outputs/supervisor_runs/`.

In [ ]:
# Supervisor progress artifact pack
paths = {
    "nhanes": OUT / "from_scratch_clinical_nhanes_summary.csv",
    "mimic": OUT / "step2_mimic_admission_summary.csv",
    "mimic_ext": OUT / "step2_mimic_admission_summary_extended.csv",
    "fusion": OUT / "step3_fusion_spec.csv",
    "reporting_lock": OUT / "final_reporting_lock.json",
    "wearable": OUT / "wearable_branch_status.json",
}
for k,p in paths.items():
    print(k, 'OK' if p.exists() else 'MISSING', p)

rows=[]
if paths["nhanes"].exists():
    rows.append(pd.read_csv(paths["nhanes"]).assign(branch='nhanes'))
if paths["mimic"].exists():
    rows.append(pd.read_csv(paths["mimic"]).assign(branch='mimic'))
if paths["mimic_ext"].exists():
    rows.append(pd.read_csv(paths["mimic_ext"]).assign(branch='mimic_extended'))

if rows:
    pack = pd.concat(rows, ignore_index=True)
    pack_path = OUT / "supervisor_progress_model_pack.csv"
    pack.to_csv(pack_path, index=False)
    print("Saved:", pack_path)
    pack.sort_values(['branch','auroc'], ascending=[True, False])
else:
    print("No model summary CSVs found yet.")

progress_status = {
    "phase": "late_step2_early_step3",
    "estimated_completion_pct": 72,
    "artifacts_present": {k: bool(paths[k].exists()) for k in paths},
}
with open(OUT / "supervisor_progress_status.json", "w", encoding="utf-8") as f:
    json.dump(progress_status, f, indent=2)
print("Saved:", OUT / "supervisor_progress_status.json")

nhanes OK /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/from_scratch_clinical_nhanes_summary.csv
mimic OK /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_admission_summary.csv
mimic_ext OK /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step2_mimic_admission_summary_extended.csv
fusion OK /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/step3_fusion_spec.csv
reporting_lock OK /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/final_reporting_lock.json
wearable OK /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/wearable_branch_status.json
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outputs/supervisor_runs/supervisor_progress_model_pack.csv
Saved: /Users/md.shadmantahsin/Desktop/STUDY/Title Defence/CKD Dataset/outpu